# 10 - Genetic algorithm clustering on 10D PCA coordinates

This notebook implements the practical assignment requirement: a generic clustering approach based on a binary genetic algorithm, applied to the pAgo project data and compared against the existing KMeans result.

Input data: 10-dimensional PCA coordinates from `data/04-analysis/pca_kmeans/latest/pca_coordinates_10D.npy`.

Default input: all records from the paper Excel dataset `mbo006184236st1.xls`, already projected into the 10D PCA space.

The genetic algorithm comes from `scripts/ga_python/gapy/ga.py`.

In [1]:
# =============================================================================
# CELL 1 - Imports
# =============================================================================

from __future__ import annotations

import html
import importlib.util
import json
import math
import shutil
import sys
import tempfile
import time
from collections import deque
from contextlib import redirect_stdout
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import DisplayHandle, HTML
from plotly.subplots import make_subplots
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)


In [2]:
# =============================================================================
# CELL 2 - Resolve project root
# =============================================================================

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Programming\Python\pAgo-project


In [3]:
# =============================================================================
# CELL 3 - Import project helpers and GA implementation
# =============================================================================

from src.pago_pipeline.ncbi_snapshot import _replace_latest_directory
from src.pago_pipeline.storage import read_json_file, sha256_of_file, write_json_atomic

GA_MODULE_PATH = PROJECT_ROOT / "scripts" / "ga_python" / "gapy" / "ga.py"
if not GA_MODULE_PATH.exists():
    raise FileNotFoundError(f"GA implementation not found: {GA_MODULE_PATH}")

ga_spec = importlib.util.spec_from_file_location("gapy_ga", GA_MODULE_PATH)
if ga_spec is None or ga_spec.loader is None:
    raise RuntimeError(f"Could not load GA module from {GA_MODULE_PATH}")

gapy_ga = importlib.util.module_from_spec(ga_spec)
ga_spec.loader.exec_module(gapy_ga)
gago = gapy_ga.gago

print(f"Loaded GA module: {GA_MODULE_PATH}")


Loaded GA module: C:\Programming\Python\pAgo-project\scripts\ga_python\gapy\ga.py


## Method

The GA uses a fixed upper bound for the number of cluster slots, but the effective number of clusters is discovered by the active centroid bits.

Chromosome representation:

- each candidate cluster slot has one activation bit;
- each active slot stores one centroid in the 10D PCA space;
- each centroid coordinate is encoded as one byte and mapped to the observed PCA coordinate range;
- each record is assigned to the nearest active centroid.

Fitness is minimized and combines:

- normalized within-cluster sum of squares;
- penalty for tiny or empty clusters;
- small penalty for using many clusters;
- separation penalty when centroids are too close.

In [4]:
# =============================================================================
# CELL 4 | Configuration
# =============================================================================

import html
import sys
import time

from collections import deque
from pathlib import Path
from typing import Any

import numpy as np

from IPython.display import DisplayHandle, HTML


PCA_KMEANS_LATEST_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "04-analysis"
    / "pca_kmeans"
    / "latest"
)

FILTERED_DATASETS_LATEST_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "03-features"
    / "pago_qc"
    / "filtered_datasets"
    / "latest"
)

GA_CLUSTERING_ROOT_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "04-analysis"
    / "ga_clustering"
)


# =============================================================================
# Input dataset
# =============================================================================

INPUT_DATASET_NAME = "paper_excel_mbo006184236st1_all"

FILTERED_DATASET_FILE_BY_NAME = {
    "classic_pago_high_precision": "classic_pago_high_precision_records.csv",
    "classic_pago_review": "classic_pago_review_records.csv",
    "excluded": "excluded_records.csv",
    "piwi_re": "piwi_re_records.csv",
}


# =============================================================================
# PCA representation
# =============================================================================

PCA_COMPONENT_COUNT = 10
PLOT_COMPONENT_COUNT = 3



# =============================================================================
# GA clustering search space
# =============================================================================

MAX_CLUSTER_COUNT = 30
MIN_CLUSTER_COUNT = 2
MIN_CLUSTER_SIZE = 5


# =============================================================================
# Fitness weights
# =============================================================================

EMPTY_CLUSTER_PENALTY_WEIGHT = 2.0
SMALL_CLUSTER_PENALTY_WEIGHT = 0.5
CLUSTER_COUNT_PENALTY_WEIGHT = 0.08
SEPARATION_PENALTY_WEIGHT = 0.05


# =============================================================================
# GA execution
# =============================================================================

GA_OPTIONS = {
    "PopulationSize": 1000,
    "Generations": 200000,
    "MutationRate": 0.01,
    "EliteCount": 4,
    "SelectionPressure": 1.5,
    "CrossoverRate": 1.0,
    "RandomState": 42,
    "Verbose": True,
}


# =============================================================================
# Runtime telemetry
# =============================================================================

ESTIMATED_CPU_MAX_POWER_WATTS = 65.0


# =============================================================================
# Animation generation selection
# =============================================================================

# The animation represents:
#
# 1. the first recorded generation
# 2. every strict improvement of the best fitness
# 3. the final recorded generation

GA_ANIMATION_GENERATION_SELECTION_MODE = "improvements_plus_endpoints"
GA_ANIMATION_IMPROVEMENT_EPSILON = 0.0

GA_ANIMATION_MAX_SELECTED_GENERATIONS = 60

# Evita um slider gigantesco.
GA_ANIMATION_MAX_SLIDER_STEPS = 60


# =============================================================================
# Animation transition logic
# =============================================================================

# Each selected generation is shown cluster by cluster.
#
# For each focused cluster:
#
# 1. all current members are displayed together
# 2. stable members already assigned to the focused centroid use its color
# 3. incoming members initially retain their previous centroid color
# 4. incoming members change color one by one
#
# Stable members are never traversed individually.

GA_ANIMATION_TRANSITION_MODE = "incoming_members_only"
GA_ANIMATION_INCOMING_POINT_ORDER = "ascending_distance_to_target_centroid"

# Safety limit for accidental generation of excessively large animations.
# Set to None to disable the limit.
GA_ANIMATION_MAX_TOTAL_FRAMES = 100000


# =============================================================================
# Animation playback
# =============================================================================

GA_ANIMATION_BASE_FRAME_DURATION_MS = 220
GA_ANIMATION_MIN_FRAME_DURATION_MS = 0.05
GA_ANIMATION_TRANSITION_MS = 0
GA_ANIMATION_DEFAULT_SPEED_MULTIPLIER = 2.0

GA_ANIMATION_PLAYBACK_SPEED_MULTIPLIERS = (
    1.0,
    2.0,
    4.0,
    8.0,
    16.0,
    32.0,
    64.0,
    128.0,
    256.0,
    512.0,
    1000.0,
)


# =============================================================================
# Animation visual encoding
# =============================================================================

GA_ANIMATION_ROTATE_CAMERA = False

GA_ANIMATION_CAMERA_EYE_X = 1.85
GA_ANIMATION_CAMERA_EYE_Y = 0.00
GA_ANIMATION_CAMERA_EYE_Z = 1.15

GA_ANIMATION_AXIS_PADDING_FRACTION = 0.05

# Reduced point sizes.
GA_ANIMATION_OTHER_POINT_SIZE = 1.55
GA_ANIMATION_FOCUS_POINT_SIZE = 1.85
GA_ANIMATION_TRANSFER_POINT_SIZE = 3.10

GA_ANIMATION_OTHER_CENTROID_SIZE = 6.20
GA_ANIMATION_FOCUS_CENTROID_SIZE = 8.20

# Opacity values.
#
# 1.00 means 0 percent transparency.
# 0.80 means 20 percent transparency.
# 0.50 means 50 percent transparency.

GA_ANIMATION_OTHER_POINT_OPACITY = 0.50
GA_ANIMATION_FOCUS_POINT_OPACITY = 0.80
GA_ANIMATION_TRANSFER_POINT_OPACITY = 1.00
GA_ANIMATION_OTHER_CENTROID_OPACITY = 1.00
GA_ANIMATION_FOCUS_CENTROID_OPACITY = 1.00
GA_ANIMATION_DISTANCE_LINE_OPACITY = 1.00
GA_ANIMATION_DISTANCE_LINE_WIDTH = 1

# Persistent centroid IDs are inferred through Hungarian assignment
# maximizing shared membership between consecutive selected generations.
GA_ANIMATION_CLUSTER_IDENTITY_TRACKING = "membership_overlap_hungarian"


# =============================================================================
# Verbose live output
# =============================================================================

GA_VERBOSE_LIVE_LINE_COUNT = 10
GA_VERBOSE_LIVE_UPDATE_INTERVAL_SECONDS = 0.25
GA_VERBOSE_LOG_PREVIEW_HEAD_LINES = 0
GA_VERBOSE_LOG_PREVIEW_TAIL_LINES = 10
GA_CONVERGENCE_IMPROVEMENT_PREVIEW_COUNT = 30
GA_STOPPING_WINDOW_GENERATIONS = 5000


# =============================================================================
# Verbose output helpers
# =============================================================================

def print_log_preview(
    log_path: Path,
    *,
    head_lines: int,
    tail_lines: int,
) -> None:
    if not log_path.exists():
        print(f"Verbose log not found: {log_path}")
        return

    lines = log_path.read_text(
        encoding="utf-8",
    ).splitlines()

    print(
        f"Verbose GA log preview "
        f"({len(lines)} total lines):"
    )

    if len(lines) <= head_lines + tail_lines:
        for line in lines:
            print(line)

        return

    for line in lines[:head_lines]:
        print(line)

    omitted_count = (
        len(lines)
        - head_lines
        - tail_lines
    )

    print(
        f"... {omitted_count} lines omitted. "
        f"Full log is saved to {log_path}"
    )

    for line in lines[-tail_lines:]:
        print(line)


class LiveGAVerboseLogger:
    def __init__(
        self,
        log_file,
        *,
        log_path: Path,
        visible_line_count: int,
        update_interval_seconds: float,
        display_stdout,
    ) -> None:
        self.log_file = log_file
        self.log_path = log_path
        self.display_stdout = display_stdout

        self.visible_lines = deque(
            maxlen=visible_line_count,
        )

        self.update_interval_seconds = float(
            update_interval_seconds,
        )

        self.partial_line = ""
        self.last_rendered_at = 0.0
        self.is_rendering = False
        self.display_handle = None

    def write(
        self,
        text: str,
    ) -> int:
        self.log_file.write(text)
        self.partial_line += text

        while "\n" in self.partial_line:
            line, self.partial_line = (
                self.partial_line.split(
                    "\n",
                    1,
                )
            )

            self.visible_lines.append(line)
            self.render(force=False)

        return len(text)

    def flush(
        self,
    ) -> None:
        self.log_file.flush()

    def render(
        self,
        *,
        force: bool,
    ) -> None:
        if self.is_rendering:
            return

        now = time.perf_counter()

        if (
            not force
            and (
                now
                - self.last_rendered_at
            )
            < self.update_interval_seconds
        ):
            return

        self.last_rendered_at = now
        self.log_file.flush()

        rendered_lines = [
            (
                "GA running. Showing last "
                f"{self.visible_lines.maxlen} "
                "verbose lines."
            ),
            f"Full verbose log: {self.log_path}",
            "=" * 88,
            *self.visible_lines,
        ]

        rendered_text = "\n".join(
            rendered_lines,
        )

        html_output = HTML(
            f"<pre>{html.escape(rendered_text)}</pre>",
        )

        current_stdout = sys.stdout
        self.is_rendering = True

        try:
            sys.stdout = self.display_stdout

            if self.display_handle is None:
                self.display_handle = (
                    DisplayHandle()
                )

                self.display_handle.display(
                    html_output,
                )

            else:
                self.display_handle.update(
                    html_output,
                )

        finally:
            sys.stdout = current_stdout
            self.is_rendering = False


def print_convergence_improvements(
    history: dict[str, Any],
    *,
    max_rows: int,
) -> None:
    best_fitness = np.asarray(
        history[
            "best_fitness_history"
        ],
        dtype=float,
    )

    if best_fitness.size == 0:
        print(
            "No convergence history available.",
        )

        return

    improvement_mask = np.r_[
        True,
        (
            np.diff(
                best_fitness,
            )
            < 0.0
        ),
    ]

    improvement_indices = np.flatnonzero(
        improvement_mask,
    )

    selected_indices = (
        improvement_indices[
            -max_rows:
        ]
    )

    print(
        "Best fitness improvements "
        f"({len(improvement_indices)} total, "
        f"showing last "
        f"{len(selected_indices)}):"
    )

    for index in selected_indices:
        print(
            f"Geração {index + 1:6d} | "
            f"Melhor fitness = "
            f"{best_fitness[index]:.10f}"
        )


print(
    "PCA and KMeans latest directory: "
    f"{PCA_KMEANS_LATEST_DIRECTORY}"
)

print(
    "Filtered datasets latest directory: "
    f"{FILTERED_DATASETS_LATEST_DIRECTORY}"
)

print(
    "GA clustering output root: "
    f"{GA_CLUSTERING_ROOT_DIRECTORY}"
)

print(
    f"Input dataset: "
    f"{INPUT_DATASET_NAME}"
)

print(
    f"GA options: "
    f"{GA_OPTIONS}"
)


PCA and KMeans latest directory: C:\Programming\Python\pAgo-project\data\04-analysis\pca_kmeans\latest
Filtered datasets latest directory: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\filtered_datasets\latest
GA clustering output root: C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering
Input dataset: paper_excel_mbo006184236st1_all
GA options: {'PopulationSize': 1000, 'Generations': 200000, 'MutationRate': 0.01, 'EliteCount': 4, 'SelectionPressure': 1.5, 'CrossoverRate': 1.0, 'RandomState': 42, 'Verbose': True}


In [5]:
# =============================================================================
# CELL 5 - Load PCA coordinates, KMeans labels, and optional notebook-09 subset
# =============================================================================

pca_manifest = read_json_file(input_file_path=PCA_KMEANS_LATEST_DIRECTORY / "manifest.json")
pca_coordinates_file_path = PCA_KMEANS_LATEST_DIRECTORY / pca_manifest["pca_coordinates_file_name"]
explained_variance_file_path = PCA_KMEANS_LATEST_DIRECTORY / pca_manifest["explained_variance_ratio_file_name"]
cluster_assignments_file_path = PCA_KMEANS_LATEST_DIRECTORY / pca_manifest["cluster_assignments_file_name"]

pca_coordinates = np.load(pca_coordinates_file_path)
explained_variance_ratio = np.load(explained_variance_file_path)
cluster_assignments_df = pd.read_csv(cluster_assignments_file_path, low_memory=False)

if pca_coordinates.ndim != 2 or pca_coordinates.shape[1] < PCA_COMPONENT_COUNT:
    raise RuntimeError(
        f"Expected PCA coordinates with at least {PCA_COMPONENT_COUNT} columns, "
        f"got shape {pca_coordinates.shape}."
    )
if len(cluster_assignments_df) != pca_coordinates.shape[0]:
    raise RuntimeError(
        "PCA coordinate row count does not match cluster assignments row count."
    )

analysis_df = cluster_assignments_df.copy()
for component_index in range(PCA_COMPONENT_COUNT):
    analysis_df[f"pc{component_index + 1}"] = pca_coordinates[:, component_index]

ALL_PAPER_DATASET_NAMES = {"all_pca_records", "paper_excel_mbo006184236st1_all"}

if INPUT_DATASET_NAME in ALL_PAPER_DATASET_NAMES:
    selected_df = analysis_df.copy()
else:
    if INPUT_DATASET_NAME not in FILTERED_DATASET_FILE_BY_NAME:
        raise ValueError(f"Unknown INPUT_DATASET_NAME: {INPUT_DATASET_NAME}")
    filtered_file_path = FILTERED_DATASETS_LATEST_DIRECTORY / FILTERED_DATASET_FILE_BY_NAME[INPUT_DATASET_NAME]
    filtered_df = pd.read_csv(filtered_file_path, low_memory=False, dtype={"protein_uid": "string"})
    selected_uids = set(filtered_df["protein_uid"].astype("string"))
    selected_df = analysis_df[analysis_df["protein_uid"].astype("string").isin(selected_uids)].copy()

if len(selected_df) < 3:
    raise RuntimeError(f"Selected dataset is too small for clustering: {len(selected_df)} rows.")

selected_df = selected_df.reset_index(drop=True)
feature_columns = [f"pc{component_index + 1}" for component_index in range(PCA_COMPONENT_COUNT)]
plot_columns = [f"pc{component_index + 1}" for component_index in range(PLOT_COMPONENT_COUNT)]
X = selected_df[feature_columns].to_numpy(dtype=float)
kmeans_labels = selected_df["cluster_label"].to_numpy(dtype=int)

print(f"Loaded PCA coordinates: {pca_coordinates.shape}")
print(f"Selected rows for GA clustering: {X.shape[0]}")
print(f"Feature matrix used by GA: {X.shape}")
print(f"Existing KMeans cluster count in selected rows: {pd.Series(kmeans_labels).nunique()}")
print(f"Explained variance in first {PCA_COMPONENT_COUNT} PCs: {explained_variance_ratio[:PCA_COMPONENT_COUNT].sum():.6f}")


Loaded PCA coordinates: (1010, 10)
Selected rows for GA clustering: 1010
Feature matrix used by GA: (1010, 10)
Existing KMeans cluster count in selected rows: 10
Explained variance in first 10 PCs: 0.151838


In [6]:
# =============================================================================
# CELL 6 - GA chromosome decoding and fitness function
# =============================================================================

@dataclass(frozen=True)
class DecodedClustering:
    active_centroids: np.ndarray
    labels: np.ndarray
    min_squared_distances: np.ndarray
    effective_cluster_count: int
    empty_active_cluster_count: int
    cluster_sizes: np.ndarray


feature_min = X.min(axis=0)
feature_max = X.max(axis=0)
feature_span = np.where((feature_max - feature_min) == 0.0, 1.0, feature_max - feature_min)
global_center = X.mean(axis=0)
total_sse = float(np.sum((X - global_center) ** 2))
if total_sse <= 0.0:
    raise RuntimeError("Input features have zero variance; clustering is not meaningful.")

coordinate_bits = 8
coordinate_levels = (2 ** coordinate_bits) - 1
bits_per_centroid = 1 + PCA_COMPONENT_COUNT * coordinate_bits
chromosome_bit_count = MAX_CLUSTER_COUNT * bits_per_centroid
data_scale = float(np.sqrt(total_sse / max(len(X), 1)))
x_squared_norms = np.sum(X * X, axis=1)[:, None]
coordinate_bit_weights = (1 << np.arange(coordinate_bits - 1, -1, -1, dtype=np.uint16)).astype(np.uint16)


def decode_centroid_chromosome(individual: np.ndarray) -> np.ndarray:
    individual = np.asarray(individual, dtype=np.uint8)
    if individual.shape[0] != chromosome_bit_count:
        raise ValueError(f"Expected {chromosome_bit_count} bits, got {individual.shape[0]}.")

    centroid_blocks = individual.reshape(MAX_CLUSTER_COUNT, bits_per_centroid)
    active_mask = centroid_blocks[:, 0].astype(bool)
    if not np.any(active_mask):
        return np.empty((0, PCA_COMPONENT_COUNT), dtype=float)

    raw_coordinates = (
        centroid_blocks[active_mask, 1:]
        .reshape(-1, PCA_COMPONENT_COUNT, coordinate_bits)
        .astype(np.uint16)
        @ coordinate_bit_weights
    ).astype(float)
    return feature_min + (raw_coordinates / coordinate_levels) * feature_span


def assign_points_to_centroids(features: np.ndarray, centroids: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    feature_squared_norms = x_squared_norms if features is X else np.sum(features * features, axis=1)[:, None]
    squared_distances = (
        feature_squared_norms
        + np.sum(centroids * centroids, axis=1)[None, :]
        - 2.0 * features @ centroids.T
    )
    np.maximum(squared_distances, 0.0, out=squared_distances)
    labels = np.argmin(squared_distances, axis=1).astype(int)
    min_squared_distances = squared_distances[np.arange(features.shape[0]), labels]
    return labels, min_squared_distances


def decode_clustering(individual: np.ndarray) -> DecodedClustering:
    active_centroids = decode_centroid_chromosome(individual)
    if active_centroids.shape[0] == 0:
        return DecodedClustering(
            active_centroids,
            np.zeros(len(X), dtype=int),
            np.full(len(X), np.inf, dtype=float),
            0,
            0,
            np.array([], dtype=int),
        )

    labels, min_squared_distances = assign_points_to_centroids(X, active_centroids)
    cluster_sizes = np.bincount(labels, minlength=active_centroids.shape[0])
    non_empty_mask = cluster_sizes > 0
    empty_count = int(np.sum(~non_empty_mask))

    if empty_count > 0:
        compact_centroids = active_centroids[non_empty_mask]
        labels, min_squared_distances = assign_points_to_centroids(X, compact_centroids)
        cluster_sizes = np.bincount(labels, minlength=compact_centroids.shape[0])
        active_centroids = compact_centroids

    return DecodedClustering(
        active_centroids=active_centroids,
        labels=labels,
        min_squared_distances=min_squared_distances,
        effective_cluster_count=int(active_centroids.shape[0]),
        empty_active_cluster_count=empty_count,
        cluster_sizes=cluster_sizes,
    )


def centroid_separation_penalty(centroids: np.ndarray) -> float:
    if centroids.shape[0] < 2:
        return 1.0
    centroid_squared_norms = np.sum(centroids * centroids, axis=1)
    squared_distances = centroid_squared_norms[:, None] + centroid_squared_norms[None, :] - 2.0 * centroids @ centroids.T
    np.maximum(squared_distances, 0.0, out=squared_distances)
    distances = np.sqrt(squared_distances)
    upper_triangle = distances[np.triu_indices(centroids.shape[0], k=1)]
    min_distance = float(np.min(upper_triangle)) if upper_triangle.size else 0.0
    return 1.0 / (1.0 + (min_distance / (data_scale + 1e-12)))


def clustering_fitness(individual: np.ndarray) -> float:
    decoded = decode_clustering(individual)
    k = decoded.effective_cluster_count
    if k < MIN_CLUSTER_COUNT:
        return 1_000.0 + (MIN_CLUSTER_COUNT - k)

    within_fraction = float(np.sum(decoded.min_squared_distances) / total_sse)
    small_cluster_count = int(np.sum(decoded.cluster_sizes < MIN_CLUSTER_SIZE))

    return float(
        within_fraction
        + EMPTY_CLUSTER_PENALTY_WEIGHT * (decoded.empty_active_cluster_count / MAX_CLUSTER_COUNT)
        + SMALL_CLUSTER_PENALTY_WEIGHT * (small_cluster_count / MAX_CLUSTER_COUNT)
        + CLUSTER_COUNT_PENALTY_WEIGHT * (k / MAX_CLUSTER_COUNT)
        + SEPARATION_PENALTY_WEIGHT * centroid_separation_penalty(decoded.active_centroids)
    )


print(f"Bits per centroid slot: {bits_per_centroid}")
print(f"Chromosome bit count: {chromosome_bit_count}")
print(f"Maximum cluster slots: {MAX_CLUSTER_COUNT}")
print(f"Minimum cluster count: {MIN_CLUSTER_COUNT}")


Bits per centroid slot: 81
Chromosome bit count: 2430
Maximum cluster slots: 30
Minimum cluster count: 2


In [7]:
# =============================================================================
# CELL 7 - Run genetic algorithm
# =============================================================================

ga_run_started_at_utc = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
ga_verbose_log_path = None
if GA_OPTIONS.get("Verbose", False):
    ga_log_directory = GA_CLUSTERING_ROOT_DIRECTORY / "logs"
    ga_log_directory.mkdir(parents=True, exist_ok=True)
    ga_verbose_log_path = ga_log_directory / f"{ga_run_started_at_utc.replace(':', '-')}_ga_stdout.log"

started_at = time.perf_counter()
if ga_verbose_log_path is None:
    best_individual, final_population, final_fitness, ga_history = gago(
        clustering_fitness,
        chromosome_bit_count,
        GA_OPTIONS,
    )
else:
    with ga_verbose_log_path.open("w", encoding="utf-8") as ga_verbose_log_file:
        live_ga_logger = LiveGAVerboseLogger(
            ga_verbose_log_file,
            log_path=ga_verbose_log_path,
            visible_line_count=GA_VERBOSE_LIVE_LINE_COUNT,
            update_interval_seconds=GA_VERBOSE_LIVE_UPDATE_INTERVAL_SECONDS,
            display_stdout=sys.stdout,
        )
        with redirect_stdout(live_ga_logger):
            best_individual, final_population, final_fitness, ga_history = gago(
                clustering_fitness,
                chromosome_bit_count,
                GA_OPTIONS,
            )
        live_ga_logger.render(force=True)
        live_ga_logger.flush()
elapsed_seconds = time.perf_counter() - started_at

best_decoded = decode_clustering(best_individual)
ga_labels = best_decoded.labels.astype(int)

print("GA finished.")
print(f"Elapsed seconds: {elapsed_seconds:.3f}")
print(f"Best fitness: {float(final_fitness[0]):.10f}")
print(f"Effective GA cluster count: {best_decoded.effective_cluster_count}")
print(f"GA cluster sizes: {best_decoded.cluster_sizes.tolist()}")
if ga_verbose_log_path is not None:
    print(f"Verbose GA log: {ga_verbose_log_path}")
    print_log_preview(
        ga_verbose_log_path,
        head_lines=GA_VERBOSE_LOG_PREVIEW_HEAD_LINES,
        tail_lines=GA_VERBOSE_LOG_PREVIEW_TAIL_LINES,
    )
print_convergence_improvements(
    ga_history,
    max_rows=GA_CONVERGENCE_IMPROVEMENT_PREVIEW_COUNT,
)


GA finished.
Elapsed seconds: 35064.285
Best fitness: 0.2892918947
Effective GA cluster count: 12
GA cluster sizes: [79, 40, 13, 24, 39, 43, 203, 67, 66, 18, 402, 16]
Verbose GA log: C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\logs\2026-06-12T19-22-18Z_ga_stdout.log
Verbose GA log preview (200000 total lines):
... 199990 lines omitted. Full log is saved to C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\logs\2026-06-12T19-22-18Z_ga_stdout.log
Geração 199991 | Melhor fitness = 0.2892918947 | Média fitness = 0.9512031348
Geração 199992 | Melhor fitness = 0.2892918947 | Média fitness = 0.9522958377
Geração 199993 | Melhor fitness = 0.2892918947 | Média fitness = 0.9581660765
Geração 199994 | Melhor fitness = 0.2892918947 | Média fitness = 0.9760773708
Geração 199995 | Melhor fitness = 0.2892918947 | Média fitness = 0.9660384885
Geração 199996 | Melhor fitness = 0.2892918947 | Média fitness = 0.9790179086
Geração 199997 | Melhor fitness = 0.289291894

In [14]:
# =============================================================================
# CELL 8 - Build GA telemetry tables and plots
# =============================================================================

telemetry_metadata = ga_history.get("telemetry_metadata", {})
logical_cpu_count = max(int(telemetry_metadata.get("logical_cpu_count") or 1), 1)

def _best_individual_active_cluster_count(
    individual: np.ndarray,
) -> int:
    centroid_blocks = np.asarray(
        individual,
        dtype=np.uint8,
    ).reshape(
        MAX_CLUSTER_COUNT,
        bits_per_centroid,
    )

    return int(
        np.count_nonzero(
            centroid_blocks[
                :,
                0,
            ]
        )
    )


history_df = pd.DataFrame(
    {
        "generation": np.arange(1, len(ga_history["best_fitness_history"]) + 1),
        "best_fitness": ga_history["best_fitness_history"],
        "mean_fitness": ga_history["mean_fitness_history"],
        "best_active_cluster_count": [
            _best_individual_active_cluster_count(
                individual
            )
            for individual
            in ga_history[
                "best_individual_history"
            ]
        ],
        "generation_elapsed_seconds": ga_history["generation_elapsed_seconds"],
        "cumulative_elapsed_seconds": ga_history["cumulative_elapsed_seconds"],
        "process_cpu_seconds_delta": ga_history["process_cpu_seconds_delta"],
        "process_cpu_seconds_cumulative": ga_history["process_cpu_seconds_cumulative"],
        "process_cpu_percent_total_capacity": ga_history["process_cpu_percent_total_capacity"],
        "process_cpu_core_percent": ga_history["process_cpu_core_percent"],
        "process_rss_mb": ga_history["process_rss_mb"],
        "system_memory_percent": ga_history["system_memory_percent"],
    }
)
history_df["cumulative_elapsed_minutes"] = history_df["cumulative_elapsed_seconds"] / 60.0
history_df["estimated_cpu_energy_joules_delta"] = (
    ESTIMATED_CPU_MAX_POWER_WATTS
    * history_df["process_cpu_seconds_delta"]
    / logical_cpu_count
)
history_df["estimated_cpu_energy_joules_cumulative"] = history_df[
    "estimated_cpu_energy_joules_delta"
].cumsum()
history_df["estimated_cpu_energy_wh_cumulative"] = (
    history_df["estimated_cpu_energy_joules_cumulative"] / 3600.0
)

stopping_window = int(GA_STOPPING_WINDOW_GENERATIONS)
history_df["stopping_window_generations"] = stopping_window
history_df["best_fitness_window_start"] = history_df["best_fitness"].shift(stopping_window)
history_df["fitness_delta_window"] = (
    history_df["best_fitness_window_start"] - history_df["best_fitness"]
)
history_df["fitness_relative_improvement_window"] = np.where(
    history_df["best_fitness_window_start"] != 0.0,
    history_df["fitness_delta_window"] / history_df["best_fitness_window_start"],
    np.nan,
)
history_df["elapsed_seconds_window"] = (
    history_df["cumulative_elapsed_seconds"]
    - history_df["cumulative_elapsed_seconds"].shift(stopping_window)
)
history_df["process_cpu_seconds_window"] = (
    history_df["process_cpu_seconds_cumulative"]
    - history_df["process_cpu_seconds_cumulative"].shift(stopping_window)
)
history_df["estimated_cpu_energy_joules_window"] = (
    history_df["estimated_cpu_energy_joules_cumulative"]
    - history_df["estimated_cpu_energy_joules_cumulative"].shift(stopping_window)
)
history_df["estimated_cpu_energy_wh_window"] = (
    history_df["estimated_cpu_energy_joules_window"] / 3600.0
)
positive_window_improvement = history_df["fitness_delta_window"] > 0.0
history_df["wall_seconds_per_fitness_delta_window"] = np.where(
    positive_window_improvement,
    history_df["elapsed_seconds_window"] / history_df["fitness_delta_window"],
    np.nan,
)
history_df["process_cpu_seconds_per_fitness_delta_window"] = np.where(
    positive_window_improvement,
    history_df["process_cpu_seconds_window"] / history_df["fitness_delta_window"],
    np.nan,
)
history_df["estimated_cpu_energy_joules_per_fitness_delta_window"] = np.where(
    positive_window_improvement,
    history_df["estimated_cpu_energy_joules_window"] / history_df["fitness_delta_window"],
    np.nan,
)

def _finite_float_or_none(value: Any) -> float | None:
    value = float(value)
    return value if np.isfinite(value) else None

stopping_rows = history_df.dropna(subset=["fitness_delta_window"])
if stopping_rows.empty:
    ga_stopping_progress_summary = {
        "status": "insufficient_history",
        "window_generations": stopping_window,
        "generations_available": int(len(history_df)),
        "generations_needed_for_first_assessment": max(stopping_window + 1 - len(history_df), 0),
    }
else:
    latest_stopping_row = stopping_rows.iloc[-1]
    latest_delta = float(latest_stopping_row["fitness_delta_window"])
    ga_stopping_progress_summary = {
        "status": "still_improving" if latest_delta > 0.0 else "no_recent_best_fitness_improvement",
        "window_generations": stopping_window,
        "generation": int(latest_stopping_row["generation"]),
        "window_start_generation": int(latest_stopping_row["generation"] - stopping_window),
        "best_fitness_window_start": _finite_float_or_none(latest_stopping_row["best_fitness_window_start"]),
        "best_fitness_current": _finite_float_or_none(latest_stopping_row["best_fitness"]),
        "fitness_delta_window": latest_delta,
        "fitness_relative_improvement_window": _finite_float_or_none(latest_stopping_row["fitness_relative_improvement_window"]),
        "elapsed_seconds_window": _finite_float_or_none(latest_stopping_row["elapsed_seconds_window"]),
        "process_cpu_seconds_window": _finite_float_or_none(latest_stopping_row["process_cpu_seconds_window"]),
        "estimated_cpu_energy_wh_window": _finite_float_or_none(latest_stopping_row["estimated_cpu_energy_wh_window"]),
        "wall_seconds_per_fitness_delta_window": _finite_float_or_none(latest_stopping_row["wall_seconds_per_fitness_delta_window"]),
        "process_cpu_seconds_per_fitness_delta_window": _finite_float_or_none(latest_stopping_row["process_cpu_seconds_per_fitness_delta_window"]),
        "estimated_cpu_energy_joules_per_fitness_delta_window": _finite_float_or_none(latest_stopping_row["estimated_cpu_energy_joules_per_fitness_delta_window"]),
    }

final_evaluation_process_cpu_seconds = float(ga_history.get("final_evaluation_process_cpu_seconds_delta", 0.0))
final_evaluation_estimated_cpu_energy_joules = (
    ESTIMATED_CPU_MAX_POWER_WATTS
    * final_evaluation_process_cpu_seconds
    / logical_cpu_count
)
generation_estimated_cpu_energy_joules_total = float(history_df["estimated_cpu_energy_joules_delta"].sum())

ga_runtime_summary = {
    "elapsed_seconds_total": float(elapsed_seconds),
    "generation_elapsed_seconds_mean": float(history_df["generation_elapsed_seconds"].mean()),
    "generation_elapsed_seconds_median": float(history_df["generation_elapsed_seconds"].median()),
    "generation_elapsed_seconds_max": float(history_df["generation_elapsed_seconds"].max()),
    "final_evaluation_elapsed_seconds": float(ga_history.get("final_evaluation_elapsed_seconds", 0.0)),
    "final_evaluation_process_cpu_seconds": final_evaluation_process_cpu_seconds,
    "process_cpu_seconds_total": float(history_df["process_cpu_seconds_delta"].sum() + final_evaluation_process_cpu_seconds),
    "process_rss_mb_max": float(max(history_df["process_rss_mb"].max(), ga_history.get("final_process_rss_mb", np.nan))),
    "estimated_cpu_energy_joules_generation_total": generation_estimated_cpu_energy_joules_total,
    "estimated_cpu_energy_joules_final_evaluation": float(final_evaluation_estimated_cpu_energy_joules),
    "estimated_cpu_energy_joules_total": float(generation_estimated_cpu_energy_joules_total + final_evaluation_estimated_cpu_energy_joules),
    "estimated_cpu_energy_wh_total": float((generation_estimated_cpu_energy_joules_total + final_evaluation_estimated_cpu_energy_joules) / 3600.0),
    "estimated_cpu_max_power_watts": float(ESTIMATED_CPU_MAX_POWER_WATTS),
    "logical_cpu_count": int(logical_cpu_count),
    "stopping_progress": ga_stopping_progress_summary,
}

GA_FITNESS_CONVERGENCE_Y_RANGE = [
    0.0,
    2.5,
]

GA_CHART_EARLY_TIME_WINDOW_MINUTES = 50.0

GA_CHART_EARLY_TIME_WINDOW_MINUTES = float(
    globals().get(
        "GA_CHART_EARLY_TIME_WINDOW_MINUTES",
        50.0,
    )
)
GA_CHART_FULL_TIME_RANGE = [
    0.0,
    max(
        float(history_df["cumulative_elapsed_minutes"].max()),
        GA_CHART_EARLY_TIME_WINDOW_MINUTES,
    ),
]
GA_CHART_EARLY_TIME_RANGE = [
    0.0,
    GA_CHART_EARLY_TIME_WINDOW_MINUTES,
]

convergence_fig = make_subplots(
    specs=[
        [
            {
                "secondary_y": True,
            }
        ]
    ]
)

convergence_fig.add_trace(
    go.Scatter(
        x=history_df[
            "cumulative_elapsed_minutes"
        ],
        y=history_df[
            "best_fitness"
        ],
        mode="lines",
        name="Best fitness",
        line={
            "color": "#1f77b4",
            "width": 2,
        },
    ),
    secondary_y=False,
)

convergence_fig.add_trace(
    go.Scatter(
        x=history_df[
            "cumulative_elapsed_minutes"
        ],
        y=history_df[
            "best_active_cluster_count"
        ],
        mode="lines",
        name="Active clusters",
        line={
            "color": "#2ca02c",
            "width": 2,
            "dash": "dot",
        },
    ),
    secondary_y=True,
)

convergence_fig.update_xaxes(
    title_text="Elapsed time (minutes)",
    range=GA_CHART_EARLY_TIME_RANGE,
    autorange=False,
)
convergence_fig.update_yaxes(
    title_text="Fitness",
    range=GA_FITNESS_CONVERGENCE_Y_RANGE,
    autorange=False,
    secondary_y=False,
)
convergence_fig.update_yaxes(
    title_text="Active cluster count",
    range=[
        0,
        MAX_CLUSTER_COUNT,
    ],
    autorange=False,
    secondary_y=True,
)
convergence_fig.update_layout(
    title="GA convergence over elapsed time",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "left",
        "x": 0,
    },
    updatemenus=[
        {
            "type": "buttons",
            "direction": "right",
            "x": 1.0,
            "xanchor": "right",
            "y": 1.18,
            "yanchor": "top",
            "buttons": [
                {
                    "label": "0-50 min",
                    "method": "relayout",
                    "args": [
                        {
                            "xaxis.range": GA_CHART_EARLY_TIME_RANGE,
                            "xaxis.autorange": False,
                        }
                    ],
                },
                {
                    "label": "Full",
                    "method": "relayout",
                    "args": [
                        {
                            "xaxis.range": GA_CHART_FULL_TIME_RANGE,
                            "xaxis.autorange": False,
                        }
                    ],
                },
            ],
        }
    ],
)

stopping_fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        f"Best-fitness reduction over last {stopping_window:,} generations",
        f"Relative best-fitness improvement over last {stopping_window:,} generations",
        "Wall-clock cost per additional fitness reduction",
    ),
)
stopping_fig.add_trace(
    go.Scatter(
        x=history_df["cumulative_elapsed_minutes"],
        y=history_df["fitness_delta_window"],
        name=f"Delta f ({stopping_window:,} gen)",
        mode="lines",
    ),
    row=1,
    col=1,
)
stopping_fig.add_trace(
    go.Scatter(
        x=history_df["cumulative_elapsed_minutes"],
        y=100.0 * history_df["fitness_relative_improvement_window"],
        name=f"Relative improvement ({stopping_window:,} gen)",
        mode="lines",
    ),
    row=2,
    col=1,
)
stopping_fig.add_trace(
    go.Scatter(
        x=history_df["cumulative_elapsed_minutes"],
        y=history_df["wall_seconds_per_fitness_delta_window"],
        name="Wall seconds per fitness delta",
        mode="lines",
    ),
    row=3,
    col=1,
)
stopping_fig.add_hline(y=0.0, line_dash="dash", line_color="gray", row=1, col=1)
stopping_fig.add_hline(y=0.0, line_dash="dash", line_color="gray", row=2, col=1)
stopping_fig.update_xaxes(title_text="Elapsed time (minutes)", row=3, col=1)
stopping_fig.update_yaxes(title_text="Fitness reduction", row=1, col=1)
stopping_fig.update_yaxes(title_text="Relative improvement (%)", row=2, col=1)
stopping_fig.update_yaxes(title_text="s / fitness unit", row=3, col=1)
stopping_fig.update_layout(
    title="GA stopping progress over elapsed time",
    height=900,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
)

resource_fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    specs=[[{}], [{"secondary_y": True}], [{}]],
    subplot_titles=(
        "Estimated CPU energy",
        "Memory",
        "CPU usage",
    ),
)
resource_fig.add_trace(
    go.Scatter(
        x=history_df["cumulative_elapsed_minutes"],
        y=history_df["estimated_cpu_energy_wh_cumulative"],
        name="Estimated CPU energy (Wh)",
        mode="lines",
    ),
    row=1,
    col=1,
)
resource_fig.add_trace(
    go.Scatter(
        x=history_df["cumulative_elapsed_minutes"],
        y=history_df["process_rss_mb"],
        name="Process RSS (MB)",
        mode="lines",
    ),
    row=2,
    col=1,
    secondary_y=False,
)
resource_fig.add_trace(
    go.Scatter(
        x=history_df["cumulative_elapsed_minutes"],
        y=history_df["system_memory_percent"],
        name="System memory (%)",
        mode="lines",
    ),
    row=2,
    col=1,
    secondary_y=True,
)
resource_fig.add_trace(
    go.Scatter(
        x=history_df["cumulative_elapsed_minutes"],
        y=history_df["process_cpu_percent_total_capacity"],
        name="Process CPU (% total capacity)",
        mode="lines",
    ),
    row=3,
    col=1,
)
resource_fig.add_trace(
    go.Scatter(
        x=history_df["cumulative_elapsed_minutes"],
        y=history_df["process_cpu_core_percent"],
        name="Process CPU (% of one core)",
        mode="lines",
    ),
    row=3,
    col=1,
)
resource_fig.update_xaxes(
    title_text="Elapsed time (minutes)",
    range=GA_CHART_EARLY_TIME_RANGE,
    autorange=False,
)
resource_fig.update_yaxes(title_text="Wh", row=1, col=1)
resource_fig.update_yaxes(title_text="MB", row=2, col=1, secondary_y=False)
resource_fig.update_yaxes(title_text="%", row=2, col=1, secondary_y=True)
resource_fig.update_yaxes(title_text="%", row=3, col=1)
resource_fig.update_layout(
    title="GA resource telemetry over elapsed time",
    height=900,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "left", "x": 0},
    updatemenus=[
        {
            "type": "buttons",
            "direction": "right",
            "x": 1.0,
            "xanchor": "right",
            "y": 1.12,
            "yanchor": "top",
            "buttons": [
                {
                    "label": "0-50 min",
                    "method": "relayout",
                    "args": [
                        {
                            "xaxis.range": GA_CHART_EARLY_TIME_RANGE,
                            "xaxis2.range": GA_CHART_EARLY_TIME_RANGE,
                            "xaxis3.range": GA_CHART_EARLY_TIME_RANGE,
                            "xaxis.autorange": False,
                            "xaxis2.autorange": False,
                            "xaxis3.autorange": False,
                        }
                    ],
                },
                {
                    "label": "Full",
                    "method": "relayout",
                    "args": [
                        {
                            "xaxis.range": GA_CHART_FULL_TIME_RANGE,
                            "xaxis2.range": GA_CHART_FULL_TIME_RANGE,
                            "xaxis3.range": GA_CHART_FULL_TIME_RANGE,
                            "xaxis.autorange": False,
                            "xaxis2.autorange": False,
                            "xaxis3.autorange": False,
                        }
                    ],
                },
            ],
        }
    ],
)

print("GA telemetry summary:")
print(json.dumps(ga_runtime_summary, indent=2, ensure_ascii=False))
print("GA stopping progress summary:")
print(json.dumps(ga_stopping_progress_summary, indent=2, ensure_ascii=False))
history_df.head()


GA telemetry summary:
{
  "elapsed_seconds_total": 35064.28460760001,
  "generation_elapsed_seconds_mean": 0.17532067254200004,
  "generation_elapsed_seconds_median": 0.16155109999817796,
  "generation_elapsed_seconds_max": 1.1096533999952953,
  "final_evaluation_elapsed_seconds": 0.14262610001605935,
  "final_evaluation_process_cpu_seconds": 0.15625,
  "process_cpu_seconds_total": 33591.96875,
  "process_rss_mb_max": 691.765625,
  "estimated_cpu_energy_joules_generation_total": 90977.82552083333,
  "estimated_cpu_energy_joules_final_evaluation": 0.4231770833333333,
  "estimated_cpu_energy_joules_total": 90978.24869791666,
  "estimated_cpu_energy_wh_total": 25.271735749421293,
  "estimated_cpu_max_power_watts": 65.0,
  "logical_cpu_count": 24,
  "stopping_progress": {
    "status": "still_improving",
    "window_generations": 5000,
    "generation": 200000,
    "window_start_generation": 195000,
    "best_fitness_window_start": 0.28938145565319784,
    "best_fitness_current": 0.2892918

,generation,best_fitness,mean_fitness,best_active_cluster_count,generation_elapsed_seconds,cumulative_elapsed_seconds,process_cpu_seconds_delta,process_cpu_seconds_cumulative,process_cpu_percent_total_capacity,process_cpu_core_percent,...,best_fitness_window_start,fitness_delta_window,fitness_relative_improvement_window,elapsed_seconds_window,process_cpu_seconds_window,estimated_cpu_energy_joules_window,estimated_cpu_energy_wh_window,wall_seconds_per_fitness_delta_window,process_cpu_seconds_per_fitness_delta_window,estimated_cpu_energy_joules_per_fitness_delta_window
0,1,1.867297,3.275586,14,0.243354,0.243354,0.234375,0.234375,4.012936,96.310472,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1.787548,3.129078,12,0.256963,0.500317,0.250000,0.484375,4.053758,97.290196,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1.420815,3.012456,12,0.232658,0.732975,0.234375,0.718750,4.197409,100.737820,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1.420815,2.846301,12,0.853064,1.586040,0.281250,1.000000,1.373724,32.969371,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1.414342,2.693858,12,0.228169,1.814209,0.234375,1.234375,4.279998,102.719959,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# =============================================================================
# CELL 9 - Evaluate GA and compare with existing KMeans labels
# =============================================================================

def compute_clustering_metrics(features: np.ndarray, labels: np.ndarray, *, method_name: str) -> dict[str, Any]:
    unique_labels = np.unique(labels)
    k = int(unique_labels.size)
    cluster_sizes = pd.Series(labels).value_counts().sort_index()
    metrics: dict[str, Any] = {
        "method": method_name,
        "record_count": int(features.shape[0]),
        "cluster_count": k,
        "min_cluster_size": int(cluster_sizes.min()),
        "max_cluster_size": int(cluster_sizes.max()),
    }
    if 1 < k < features.shape[0]:
        metrics["silhouette"] = float(silhouette_score(features, labels))
        metrics["davies_bouldin"] = float(davies_bouldin_score(features, labels))
        metrics["calinski_harabasz"] = float(calinski_harabasz_score(features, labels))
    else:
        metrics["silhouette"] = np.nan
        metrics["davies_bouldin"] = np.nan
        metrics["calinski_harabasz"] = np.nan
    return metrics


ga_metrics = compute_clustering_metrics(X, ga_labels, method_name="genetic_algorithm")
kmeans_metrics = compute_clustering_metrics(X, kmeans_labels, method_name="kmeans_baseline")
comparison_ari = float(adjusted_rand_score(kmeans_labels, ga_labels))

metrics_df = pd.DataFrame([ga_metrics, kmeans_metrics])
metrics_df["ari_against_other_method"] = [comparison_ari, comparison_ari]

print(f"Adjusted Rand Index between GA and KMeans: {comparison_ari:.6f}")
metrics_df


Adjusted Rand Index between GA and KMeans: 0.929675


,method,record_count,cluster_count,min_cluster_size,max_cluster_size,silhouette,davies_bouldin,calinski_harabasz,ari_against_other_method
0,genetic_algorithm,1010,12,13,402,0.419374,0.651399,415.978918,0.929675
1,kmeans_baseline,1010,10,13,403,0.390364,0.808067,289.737134,0.929675


In [16]:
# =============================================================================
# CELL 10 - Build 3D plot of PCA coordinates colored by GA clusters
# =============================================================================

plot_df = selected_df.copy()
plot_df["ga_cluster_label"] = ga_labels.astype(str)
plot_df["kmeans_cluster_label"] = kmeans_labels.astype(str)

hover_columns = [
    column
    for column in [
        "protein_uid",
        "gbseq__accession_version",
        "gbseq__organism",
        "taxonomy__03",
        "taxonomy__04",
        "sequence_length",
        "kmeans_cluster_label",
    ]
    if column in plot_df.columns
]

fig = px.scatter_3d(
    plot_df,
    x="pc1",
    y="pc2",
    z="pc3",
    color="ga_cluster_label",
    hover_data=hover_columns,
    title=(
        "GA clustering over 10D PCA coordinates "
        f"({INPUT_DATASET_NAME}, n={len(plot_df)})"
    ),
)
fig.update_traces(marker={"size": 4, "opacity": 0.82})
fig.update_layout(
    scene={
        "xaxis_title": "PC1",
        "yaxis_title": "PC2",
        "zaxis_title": "PC3",
    },
    legend_title_text="GA cluster",
)
print("3D Plotly figure prepared. It will be written to ga_clustering_3d.html in the next cell.")


3D Plotly figure prepared. It will be written to ga_clustering_3d.html in the next cell.


In [17]:
# =============================================================================
# CELL 11 | Animated GA clustering dashboard
# =============================================================================

import html
import json
import math
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import linear_sum_assignment


T_OTHER_POINTS = 0
T_FOCUS_POINTS = 1
T_PENDING_POINTS = 2
T_TRANSFER_POINT = 3
T_OTHER_CENTROIDS = 4
T_FOCUS_CENTROID = 5
T_DISTANCE_LINE = 6
T_BEST = 7
T_ACTIVE_CLUSTERS = 8
T_ENERGY = 9
T_CPU = 10
T_RAM = 11


@dataclass(frozen=True)
class AnimationState:
    history_index: int
    generation: int
    labels: np.ndarray
    centroids_3d: np.ndarray
    centroid_ids: np.ndarray


@dataclass(frozen=True)
class AnimationStep:
    name: str
    state_position: int
    generation: int
    cluster_position: int
    cluster_count: int
    focus_cluster_id: int
    focus_centroid_position: int
    kind: str
    transfer_position: int
    transfer_count: int
    transfer_point_index: int | None
    is_generation_start: bool


def _cluster_color(cluster_id: int) -> str:
    hue = (int(cluster_id) * 137.50776405003785) % 360.0
    return f"hsl({hue:.3f}, 72%, 46%)"


def _padded_range(values, padding_fraction: float, minimum_padding: float = 1e-9):
    values = np.asarray(values, dtype=float).reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return [-1.0, 1.0]

    lower = float(values.min())
    upper = float(values.max())
    span = upper - lower
    padding = max(span * float(padding_fraction), minimum_padding)
    return [lower - padding, upper + padding]


def _active_cluster_count_from_individual(individual: np.ndarray) -> int:
    centroid_blocks = np.asarray(individual, dtype=np.uint8).reshape(
        MAX_CLUSTER_COUNT,
        bits_per_centroid,
    )
    return int(np.count_nonzero(centroid_blocks[:, 0]))


if (
    "best_active_cluster_count" not in history_df.columns
    or history_df["best_active_cluster_count"].isna().all()
):
    history_df["best_active_cluster_count"] = [
        _active_cluster_count_from_individual(individual)
        for individual in ga_history["best_individual_history"]
    ]


def _history_values(column_name: str) -> np.ndarray:
    if column_name in history_df.columns:
        return history_df[column_name].to_numpy(dtype=float)
    return np.full(len(history_df), np.nan, dtype=float)


def _selected_history_indices() -> np.ndarray:
    best = history_df["best_fitness"].to_numpy(dtype=float)
    if len(best) == 0:
        return np.array([], dtype=int)
    if len(best) == 1:
        return np.array([0], dtype=int)

    improvement_indices = (
        np.flatnonzero(np.diff(best) < -float(GA_ANIMATION_IMPROVEMENT_EPSILON))
        + 1
    )

    selected_indices = np.unique(
        np.r_[
            0,
            improvement_indices,
            len(best) - 1,
        ]
    ).astype(int)

    if len(selected_indices) > int(GA_ANIMATION_MAX_SELECTED_GENERATIONS):
        sampled_positions = (
            np.linspace(
                0,
                len(selected_indices) - 1,
                int(GA_ANIMATION_MAX_SELECTED_GENERATIONS),
            )
            .round()
            .astype(int)
        )
        selected_indices = selected_indices[sampled_positions]

    return np.unique(selected_indices)


def _decode_state(history_index: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    individual = np.asarray(
        ga_history["best_individual_history"][history_index],
        dtype=np.uint8,
    )
    centroid_blocks = individual.reshape(MAX_CLUSTER_COUNT, bits_per_centroid)
    active_mask = centroid_blocks[:, 0].astype(bool)
    centroid_ids = np.flatnonzero(active_mask).astype(int)

    if centroid_ids.size == 0:
        return (
            global_center[None, :],
            np.zeros(len(X), dtype=int),
            np.array([0], dtype=int),
        )

    raw_coordinates = (
        centroid_blocks[active_mask, 1:]
        .reshape(-1, PCA_COMPONENT_COUNT, coordinate_bits)
        .astype(np.uint16)
        @ coordinate_bit_weights
    ).astype(float)

    active_centroids = feature_min + (
        raw_coordinates / coordinate_levels
    ) * feature_span

    labels, _ = assign_points_to_centroids(X, active_centroids)
    cluster_sizes = np.bincount(labels, minlength=active_centroids.shape[0])
    non_empty_mask = cluster_sizes > 0

    if not np.all(non_empty_mask):
        active_centroids = active_centroids[non_empty_mask]
        centroid_ids = centroid_ids[non_empty_mask]
        labels, _ = assign_points_to_centroids(X, active_centroids)

    return active_centroids, labels.astype(int), centroid_ids.astype(int)


def _build_states(selected_indices: np.ndarray) -> list[AnimationState]:
    states: list[AnimationState] = []
    previous_labels = None
    previous_centroids_3d = None
    previous_centroid_ids = None
    next_cluster_id = 0

    for history_index in selected_indices:
        centroids, raw_labels, decoded_centroid_ids = _decode_state(int(history_index))
        centroids_3d = centroids[:, :PLOT_COMPONENT_COUNT]
        current_count = int(len(centroids_3d))

        if previous_labels is None:
            centroid_ids = np.arange(
                next_cluster_id,
                next_cluster_id + current_count,
                dtype=int,
            )
            next_cluster_id += current_count
        else:
            centroid_ids = np.full(current_count, -1, dtype=int)
            overlap = np.zeros((len(previous_centroid_ids), current_count), dtype=int)

            for previous_position, previous_cluster_id in enumerate(previous_centroid_ids):
                previous_mask = previous_labels == int(previous_cluster_id)
                overlap[previous_position, :] = np.bincount(
                    raw_labels[previous_mask],
                    minlength=current_count,
                )

            distance = np.sqrt(
                np.sum(
                    (
                        previous_centroids_3d[:, None, :]
                        - centroids_3d[None, :, :]
                    )
                    ** 2,
                    axis=2,
                )
            )
            max_distance = float(distance.max()) if distance.size else 0.0
            normalized_distance = (
                distance / max_distance
                if max_distance > 0.0
                else distance
            )

            previous_positions, current_positions = linear_sum_assignment(
                -overlap.astype(float) + normalized_distance * 1e-6
            )
            positive_overlap = overlap[previous_positions, current_positions] > 0
            centroid_ids[current_positions[positive_overlap]] = previous_centroid_ids[
                previous_positions[positive_overlap]
            ]
            new_positions = np.flatnonzero(centroid_ids < 0)
            centroid_ids[new_positions] = np.arange(
                next_cluster_id,
                next_cluster_id + len(new_positions),
                dtype=int,
            )
            next_cluster_id += len(new_positions)

        centroid_ids = decoded_centroid_ids
        stable_labels = centroid_ids[raw_labels]

        states.append(
            AnimationState(
                history_index=int(history_index),
                generation=int(history_df.iloc[int(history_index)]["generation"]),
                labels=stable_labels,
                centroids_3d=centroids_3d,
                centroid_ids=centroid_ids,
            )
        )

        previous_labels = stable_labels
        previous_centroids_3d = centroids_3d
        previous_centroid_ids = centroid_ids

    return states


def _ordered_incoming_indices(
    previous_labels: np.ndarray,
    state: AnimationState,
    focus_cluster_id: int,
    focus_centroid_position: int,
) -> np.ndarray:
    current_members = np.flatnonzero(state.labels == focus_cluster_id)
    incoming = current_members[previous_labels[current_members] != focus_cluster_id]

    if len(incoming) == 0:
        return incoming

    centroid = state.centroids_3d[focus_centroid_position]
    squared_distance = np.sum(
        (X[incoming, :PLOT_COMPONENT_COUNT] - centroid) ** 2,
        axis=1,
    )
    return incoming[np.argsort(squared_distance)]


def _build_steps(states: list[AnimationState]) -> list[AnimationStep]:
    steps: list[AnimationStep] = []

    for state_position, state in enumerate(states):
        previous_labels = (
            state.labels.copy()
            if state_position == 0
            else states[state_position - 1].labels
        )
        ordered_centroid_positions = np.argsort(state.centroid_ids)
        cluster_count = len(ordered_centroid_positions)

        for cluster_position, centroid_position in enumerate(ordered_centroid_positions):
            cluster_id = int(state.centroid_ids[centroid_position])
            all_incoming = _ordered_incoming_indices(
                previous_labels=previous_labels,
                state=state,
                focus_cluster_id=cluster_id,
                focus_centroid_position=int(centroid_position),
            )
            steps.append(
                AnimationStep(
                    name=f"g{state.generation:09d}_c{cluster_id:05d}_focus",
                    state_position=state_position,
                    generation=state.generation,
                    cluster_position=cluster_position,
                    cluster_count=cluster_count,
                    focus_cluster_id=cluster_id,
                    focus_centroid_position=int(centroid_position),
                    kind="cluster_focus",
                    transfer_position=0,
                    transfer_count=len(all_incoming),
                    transfer_point_index=None,
                    is_generation_start=cluster_position == 0,
                )
            )

            steps.append(
                AnimationStep(
                    name=f"g{state.generation:09d}_c{cluster_id:05d}_commit",
                    state_position=state_position,
                    generation=state.generation,
                    cluster_position=cluster_position,
                    cluster_count=cluster_count,
                    focus_cluster_id=cluster_id,
                    focus_centroid_position=int(centroid_position),
                    kind="cluster_commit",
                    transfer_position=0,
                    transfer_count=len(all_incoming),
                    transfer_point_index=None,
                    is_generation_start=False,
                )
            )

    return steps


animation_generation_indices = _selected_history_indices()
if len(animation_generation_indices) == 0:
    raise RuntimeError("GA animation requires at least one recorded generation.")

animation_clustering_states = _build_states(animation_generation_indices)
animation_steps = _build_steps(animation_clustering_states)
if len(animation_steps) == 0:
    raise RuntimeError("GA animation did not generate any step.")

if (
    GA_ANIMATION_MAX_TOTAL_FRAMES is not None
    and len(animation_steps) > int(GA_ANIMATION_MAX_TOTAL_FRAMES)
):
    raise RuntimeError(
        "GA animation exceeded GA_ANIMATION_MAX_TOTAL_FRAMES. "
        f"Generated: {len(animation_steps):,}. "
        f"Limit: {GA_ANIMATION_MAX_TOTAL_FRAMES:,}."
    )

pca_ranges = [
    _padded_range(X[:, component_index], GA_ANIMATION_AXIS_PADDING_FRACTION)
    for component_index in range(PLOT_COMPONENT_COUNT)
]
early_elapsed_window_minutes = float(
    globals().get(
        "GA_CHART_EARLY_TIME_WINDOW_MINUTES",
        50.0,
    )
)
elapsed_range = [
    0.0,
    max(
        float(_history_values("cumulative_elapsed_minutes").max()),
        early_elapsed_window_minutes,
        1e-9,
    ),
]
early_elapsed_range = [
    0.0,
    early_elapsed_window_minutes,
]
fitness_range = list(
    GA_FITNESS_CONVERGENCE_Y_RANGE
)
resource_left_range = _padded_range(
    np.column_stack([
        _history_values("estimated_cpu_energy_wh_cumulative"),
        _history_values("process_cpu_percent_total_capacity"),
    ]),
    GA_ANIMATION_AXIS_PADDING_FRACTION,
)
rss_range = _padded_range(
    _history_values("process_rss_mb"),
    GA_ANIMATION_AXIS_PADDING_FRACTION,
)


def _state(step: AnimationStep) -> AnimationState:
    return animation_clustering_states[step.state_position]


def _empty_3d_marker(name: str, size: float, opacity: float):
    return go.Scatter3d(
        x=[],
        y=[],
        z=[],
        mode="markers",
        name=name,
        marker={"size": size, "opacity": opacity},
        hoverinfo="skip",
    )


def _point_trace(indices: np.ndarray, labels: np.ndarray, name: str, size: float, opacity: float):
    return go.Scatter3d(
        x=X[indices, 0],
        y=X[indices, 1],
        z=X[indices, 2],
        mode="markers",
        name=name,
        marker={
            "size": size,
            "color": [_cluster_color(int(label)) for label in labels[indices]],
            "opacity": opacity,
        },
        hoverinfo="skip",
    )


def _centroid_traces(state: AnimationState, focus_cluster_id: int, focus_centroid_position: int):
    other_positions = np.flatnonzero(state.centroid_ids != focus_cluster_id)
    focus_centroid = state.centroids_3d[focus_centroid_position]

    return [
        go.Scatter3d(
            x=state.centroids_3d[other_positions, 0],
            y=state.centroids_3d[other_positions, 1],
            z=state.centroids_3d[other_positions, 2],
            mode="markers+text",
            name="Other centroids",
            marker={
                "size": GA_ANIMATION_OTHER_CENTROID_SIZE,
                "color": [
                    _cluster_color(int(cluster_id))
                    for cluster_id in state.centroid_ids[other_positions]
                ],
                "opacity": GA_ANIMATION_OTHER_CENTROID_OPACITY,
                "symbol": "diamond",
            },
            text=[f"C{int(cluster_id)}" for cluster_id in state.centroid_ids[other_positions]],
            textposition="top center",
            hoverinfo="text",
        ),
        go.Scatter3d(
            x=[float(focus_centroid[0])],
            y=[float(focus_centroid[1])],
            z=[float(focus_centroid[2])],
            mode="markers+text",
            name="Focused centroid",
            marker={
                "size": GA_ANIMATION_FOCUS_CENTROID_SIZE,
                "color": _cluster_color(focus_cluster_id),
                "opacity": GA_ANIMATION_FOCUS_CENTROID_OPACITY,
                "symbol": "diamond",
                "line": {"color": "#111111", "width": 3},
            },
            text=[f"C{focus_cluster_id}"],
            textposition="top center",
            hoverinfo="text",
        ),
    ]


def _chart_trace(column: str, state_position: int, name: str, color: str, mode: str = "lines"):
    selected = animation_generation_indices[: state_position + 1]
    return go.Scatter(
        x=_history_values("cumulative_elapsed_minutes")[selected],
        y=_history_values(column)[selected],
        mode=mode,
        name=name,
        line={"color": color, "width": 2},
        marker={"size": 4},
    )


def _layout_title(step: AnimationStep) -> str:
    title = (
        "GA best individual clustering animation"
        f" | generation {step.generation:,}"
        f" | focused centroid C{step.focus_cluster_id}"
        f" | cluster {step.cluster_position + 1} of {step.cluster_count}"
    )
    if step.kind == "cluster_commit":
        return title + f" | committed incoming members {step.transfer_count}"
    return title + f" | incoming members {step.transfer_count}"


initial_step = animation_steps[0]
initial_state = _state(initial_step)
initial_labels = initial_state.labels.copy()
initial_focus_cluster = initial_step.focus_cluster_id
initial_other_indices = np.flatnonzero(initial_state.labels != initial_focus_cluster)
initial_focus_indices = np.flatnonzero(initial_state.labels == initial_focus_cluster)
initial_centroid_traces = _centroid_traces(
    initial_state,
    initial_focus_cluster,
    initial_step.focus_centroid_position,
)

initial_traces = [
    _empty_3d_marker(
        "Other cluster points",
        GA_ANIMATION_OTHER_POINT_SIZE,
        GA_ANIMATION_OTHER_POINT_OPACITY,
    ),
    _empty_3d_marker(
        "Focused cluster members",
        GA_ANIMATION_FOCUS_POINT_SIZE,
        GA_ANIMATION_FOCUS_POINT_OPACITY,
    ),
    _empty_3d_marker(
        "Pending incoming members",
        GA_ANIMATION_FOCUS_POINT_SIZE,
        GA_ANIMATION_FOCUS_POINT_OPACITY,
    ),
    go.Scatter3d(
        x=[],
        y=[],
        z=[],
        mode="markers",
        name="Current incoming member",
        marker={
            "size": GA_ANIMATION_TRANSFER_POINT_SIZE,
            "opacity": GA_ANIMATION_TRANSFER_POINT_OPACITY,
            "line": {"color": "#111111", "width": 2},
        },
        hoverinfo="skip",
    ),
    initial_centroid_traces[0],
    initial_centroid_traces[1],
    go.Scatter3d(
        x=[],
        y=[],
        z=[],
        mode="lines",
        name="Current centroid distance",
        line={"width": GA_ANIMATION_DISTANCE_LINE_WIDTH},
        opacity=GA_ANIMATION_DISTANCE_LINE_OPACITY,
        hoverinfo="skip",
    ),
    _chart_trace("best_fitness", 0, "Best fitness", "#1f77b4", "lines+markers"),
    _chart_trace("best_active_cluster_count", 0, "Active clusters", "#2ca02c"),
    _chart_trace("estimated_cpu_energy_wh_cumulative", 0, "Energy estimate in Wh", "#2ca02c"),
    _chart_trace("process_cpu_percent_total_capacity", 0, "CPU usage", "#d62728"),
    _chart_trace("process_rss_mb", 0, "RAM RSS in MB", "#9467bd"),
]

animation_fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"type": "scene", "rowspan": 2}, {"type": "xy", "secondary_y": True}],
        [None, {"type": "xy", "secondary_y": True}],
    ],
    column_widths=[0.66, 0.34],
    row_heights=[0.48, 0.52],
    horizontal_spacing=0.045,
    vertical_spacing=0.09,
    subplot_titles=(
        "Best individual clusters in PCA space",
        "Convergence",
        "Resources",
    ),
)

for trace_index, trace in enumerate(initial_traces):
    if trace_index <= T_DISTANCE_LINE:
        animation_fig.add_trace(trace, row=1, col=1)
    elif trace_index == T_BEST:
        animation_fig.add_trace(trace, row=1, col=2, secondary_y=False)
    elif trace_index == T_ACTIVE_CLUSTERS:
        animation_fig.add_trace(trace, row=1, col=2, secondary_y=True)
    elif trace_index in {T_ENERGY, T_CPU}:
        animation_fig.add_trace(trace, row=2, col=2, secondary_y=False)
    else:
        animation_fig.add_trace(trace, row=2, col=2, secondary_y=True)


effective_playback_speed_multipliers = tuple(
    float(value)
    for value in globals().get(
        "GA_ANIMATION_PLAYBACK_SPEED_MULTIPLIERS",
        (),
    )
)
if (
    not effective_playback_speed_multipliers
    or max(effective_playback_speed_multipliers) < 1000.0
):
    effective_playback_speed_multipliers = (
        1.0,
        2.0,
        4.0,
        8.0,
        16.0,
        32.0,
        64.0,
        128.0,
        256.0,
        512.0,
        1000.0,
    )


def _duration_for_speed(speed_multiplier: float) -> float:
    speed = max(float(speed_multiplier), 1e-9)
    return max(
        float(GA_ANIMATION_BASE_FRAME_DURATION_MS) / speed,
        float(GA_ANIMATION_MIN_FRAME_DURATION_MS),
    )


default_animation_frame_duration_ms = _duration_for_speed(
    GA_ANIMATION_DEFAULT_SPEED_MULTIPLIER
)
frame_transition_ms = min(
    int(GA_ANIMATION_TRANSITION_MS),
    max(default_animation_frame_duration_ms - 1, 0),
)

animation_frames: list[go.Frame] = []
animation_fig.frames = ()

active_state_position = None
working_labels = None
animation_step_payloads: list[dict[str, Any]] = []

for step_position, step in enumerate(animation_steps):
    current_state = _state(step)

    if active_state_position != step.state_position:
        if active_state_position is not None:
            expected = animation_clustering_states[active_state_position].labels
            if not np.array_equal(working_labels, expected):
                raise RuntimeError(
                    "Previous transition did not converge to expected persistent labels."
                )
        active_state_position = step.state_position
        working_labels = (
            current_state.labels.copy()
            if step.state_position == 0
            else animation_clustering_states[step.state_position - 1].labels.copy()
        )

    commit_point_indices: list[int] = []
    if step.kind == "cluster_commit":
        commit_point_indices = (
            np.flatnonzero(
                (current_state.labels == step.focus_cluster_id)
                & (working_labels != step.focus_cluster_id)
            )
            .astype(int)
            .tolist()
        )
        if commit_point_indices:
            working_labels[commit_point_indices] = step.focus_cluster_id
    elif step.kind != "cluster_focus":
        raise RuntimeError(f"Unsupported animation step kind: {step.kind}")

    animation_step_payloads.append(
        {
            "name": step.name,
            "position": int(step_position),
            "state_position": int(step.state_position),
            "generation": int(step.generation),
            "cluster_position": int(step.cluster_position),
            "cluster_count": int(step.cluster_count),
            "focus_cluster_id": int(step.focus_cluster_id),
            "focus_centroid_position": int(step.focus_centroid_position),
            "kind": step.kind,
            "transfer_position": int(step.transfer_position),
            "transfer_count": int(step.transfer_count),
            "transfer_point_index": (
                None
                if step.transfer_point_index is None
                else int(step.transfer_point_index)
            ),
            "commit_point_indices": commit_point_indices,
            "is_generation_start": bool(step.is_generation_start),
        }
    )

if not np.array_equal(working_labels, animation_clustering_states[-1].labels):
    raise RuntimeError("Final transition did not converge to expected persistent labels.")


def _finite_float_list(values) -> list[float | None]:
    result: list[float | None] = []
    for value in np.asarray(values, dtype=float).reshape(-1):
        result.append(None if not np.isfinite(value) else float(value))
    return result


def _int_list(values) -> list[int]:
    return [int(value) for value in np.asarray(values, dtype=int).reshape(-1)]


def _hover_value(value) -> str:
    if pd.isna(value):
        return ""

    return html.escape(
        str(value)
    )


def _point_hover_base_texts() -> list[str]:
    hover_columns = [
        (
            "Protein UID",
            "protein_uid",
        ),
        (
            "Accession",
            "gbseq__accession_version",
        ),
        (
            "Organism",
            "gbseq__organism",
        ),
        (
            "Taxonomy 03",
            "taxonomy__03",
        ),
        (
            "Taxonomy 04",
            "taxonomy__04",
        ),
        (
            "Sequence length",
            "sequence_length",
        ),
        (
            "KMeans cluster",
            "cluster_label",
        ),
    ]

    available_columns = [
        (
            label,
            column,
        )
        for label,
        column in hover_columns
        if column
        in selected_df.columns
    ]

    hover_texts: list[str] = []

    for point_index, row in selected_df.iterrows():
        parts = [
            f"Point index {int(point_index)}"
        ]

        for label, column in available_columns:
            value = _hover_value(
                row[
                    column
                ]
            )

            if value:
                parts.append(
                    f"{label}: {value}"
                )

        hover_texts.append(
            "<br>".join(
                parts
            )
        )

    return hover_texts


all_generation_start_step_indices = [
    int(index)
    for index, step in enumerate(animation_steps)
    if step.is_generation_start
]
slider_generation_start_step_indices = all_generation_start_step_indices

if len(slider_generation_start_step_indices) > int(GA_ANIMATION_MAX_SLIDER_STEPS):
    slider_sample_positions = (
        np.linspace(
            0,
            len(slider_generation_start_step_indices) - 1,
            int(GA_ANIMATION_MAX_SLIDER_STEPS),
        )
        .round()
        .astype(int)
    )
    slider_generation_start_step_indices = [
        slider_generation_start_step_indices[int(position)]
        for position in slider_sample_positions
    ]

selected = animation_generation_indices
animation_dashboard_payload = {
    "points": {
        "x": _finite_float_list(X[:, 0]),
        "y": _finite_float_list(X[:, 1]),
        "z": _finite_float_list(X[:, 2]),
    },
    "initial_labels": _int_list(animation_clustering_states[0].labels),
    "point_hover_base_text": _point_hover_base_texts(),
    "states": [
        {
            "generation": int(state.generation),
            "centroid_ids": _int_list(state.centroid_ids),
            "centroids": [
                [float(coordinate) for coordinate in centroid]
                for centroid in state.centroids_3d
            ],
        }
        for state in animation_clustering_states
    ],
    "steps": animation_step_payloads,
    "generation_start_step_indices": slider_generation_start_step_indices,
    "charts": {
        "elapsed": _finite_float_list(_history_values("cumulative_elapsed_minutes")[selected]),
        "early_elapsed_range": _finite_float_list(early_elapsed_range),
        "full_elapsed_range": _finite_float_list(elapsed_range),
        "early_elapsed_window_minutes": float(early_elapsed_window_minutes),
        "best_fitness": _finite_float_list(_history_values("best_fitness")[selected]),
        "active_clusters": _finite_float_list(_history_values("best_active_cluster_count")[selected]),
        "energy_wh": _finite_float_list(_history_values("estimated_cpu_energy_wh_cumulative")[selected]),
        "cpu_percent": _finite_float_list(_history_values("process_cpu_percent_total_capacity")[selected]),
        "rss_mb": _finite_float_list(_history_values("process_rss_mb")[selected]),
    },
    "timing": {
        "base_frame_duration_ms": int(GA_ANIMATION_BASE_FRAME_DURATION_MS),
        "other_point_size": float(GA_ANIMATION_OTHER_POINT_SIZE),
        "focus_point_size": float(GA_ANIMATION_FOCUS_POINT_SIZE),
        "minimum_frame_duration_ms": float(GA_ANIMATION_MIN_FRAME_DURATION_MS),
        "default_speed_multiplier": float(GA_ANIMATION_DEFAULT_SPEED_MULTIPLIER),
        "distance_line_width": float(GA_ANIMATION_DISTANCE_LINE_WIDTH),
        "speed_multipliers": [
            float(value)
            for value in effective_playback_speed_multipliers
        ],
    },
}

animation_dashboard_post_script = r"""
(function() {
  const graph = document.getElementById('{plot_id}');
  const payload = __PAYLOAD__;
  const trace = {
    otherPoints: 0, focusPoints: 1, pendingPoints: 2, transferPoint: 3,
    otherCentroids: 4, focusCentroid: 5, distanceLine: 6,
    best: 7, activeClusters: 8, energy: 9, cpu: 10, ram: 11
  };
  const points = payload.points;
  const pointHoverBaseText = payload.point_hover_base_text || [];
  const steps = payload.steps;
  const states = payload.states;
  const charts = payload.charts;
  const timing = payload.timing;
  let currentStepIndex = 0;
  let playbackFrameRequest = null;
  let playbackLastTimestamp = null;
  let playbackStepRemainderMs = 0;
  let isPlaying = false;
  let inspectionMode = true;
  let currentSpeedIndex = Math.max(
    0,
    timing.speed_multipliers.findIndex(
      (speed) => Number(speed) >= Number(timing.default_speed_multiplier || 1)
    )
  );
  if (currentSpeedIndex < 0) {
    currentSpeedIndex = 0;
  }

  function clusterColor(clusterId) {
    const hue = ((Number(clusterId) * 137.50776405003785) % 360 + 360) % 360;
    return `hsl(${hue.toFixed(3)}, 72%, 46%)`;
  }

  function coordinates(indices, axis) {
    const source = points[axis];
    return indices.map((index) => source[index]);
  }

  function colorsFor(indices, labels) {
    return indices.map((index) => clusterColor(labels[index]));
  }

  function hoverTextFor(indices, labels, targetLabels) {
    return indices.map((index) => {
      const baseText = pointHoverBaseText[index] || `Point index ${index}`;
      const currentCluster = labels[index];
      const targetCluster = targetLabels[index];
      return `${baseText}<br>Current GA cluster: C${currentCluster}<br>Target GA cluster: C${targetCluster}`;
    });
  }

  function pointSize(baseSize) {
    return inspectionMode ? baseSize * 1.2 : baseSize;
  }

  function pointHoverInfo() {
    return inspectionMode ? 'text' : 'skip';
  }

  function labelsBefore(stepIndex) {
    const labels = payload.initial_labels.slice();
    for (let i = 0; i < stepIndex; i += 1) {
      const step = steps[i];
      if (step.kind === 'cluster_commit') {
        for (const pointIndex of step.commit_point_indices) {
          labels[pointIndex] = step.focus_cluster_id;
        }
      }
    }
    return labels;
  }

  function applyCurrentStep(labels, step) {
    if (step.kind === 'cluster_commit') {
      for (const pointIndex of step.commit_point_indices) {
        labels[pointIndex] = step.focus_cluster_id;
      }
    }
  }

  function targetLabelsForState(stepIndex, labels) {
    const target = labels.slice();
    const statePosition = steps[stepIndex].state_position;
    for (let i = stepIndex; i < steps.length; i += 1) {
      const step = steps[i];
      if (step.state_position !== statePosition) {
        break;
      }
      if (step.kind === 'cluster_commit') {
        for (const pointIndex of step.commit_point_indices) {
          target[pointIndex] = step.focus_cluster_id;
        }
      }
    }
    return target;
  }

  function titleFor(step) {
    const prefix = `GA best individual clustering animation | generation ${step.generation}`;
    const focus = ` | focused centroid C${step.focus_cluster_id} | cluster ${step.cluster_position + 1} of ${step.cluster_count}`;
    if (step.kind === 'cluster_commit') {
      return `${prefix}${focus} | committed incoming members ${step.transfer_count}`;
    }
    return `${prefix}${focus} | incoming members ${step.transfer_count}`;
  }

  function splitPointIndices(labels, targetLabels, focusClusterId) {
    const other = [];
    const focus = [];
    const pending = [];
    for (let index = 0; index < labels.length; index += 1) {
      if (targetLabels[index] === focusClusterId) {
        if (labels[index] === focusClusterId) {
          focus.push(index);
        } else {
          pending.push(index);
        }
      } else {
        other.push(index);
      }
    }
    return { other, focus, pending };
  }

  function restylePoints(step, labels, targetLabels) {
    const groups = splitPointIndices(labels, targetLabels, step.focus_cluster_id);
    const focusColor = clusterColor(step.focus_cluster_id);
    const hoverInfo = pointHoverInfo();
    const otherHoverText = inspectionMode ? hoverTextFor(groups.other, labels, targetLabels) : [];
    const focusHoverText = inspectionMode ? hoverTextFor(groups.focus, labels, targetLabels) : [];
    const pendingHoverText = inspectionMode ? hoverTextFor(groups.pending, labels, targetLabels) : [];
    Plotly.restyle(graph, {
      x: [coordinates(groups.other, 'x')],
      y: [coordinates(groups.other, 'y')],
      z: [coordinates(groups.other, 'z')],
      'marker.color': [colorsFor(groups.other, labels)],
      'marker.size': [pointSize(timing.other_point_size)],
      hovertext: [otherHoverText],
      hoverinfo: hoverInfo
    }, [trace.otherPoints]);
    Plotly.restyle(graph, {
      x: [coordinates(groups.focus, 'x')],
      y: [coordinates(groups.focus, 'y')],
      z: [coordinates(groups.focus, 'z')],
      'marker.color': [focusColor],
      'marker.size': [pointSize(timing.focus_point_size)],
      hovertext: [focusHoverText],
      hoverinfo: hoverInfo
    }, [trace.focusPoints]);
    Plotly.restyle(graph, {
      x: [coordinates(groups.pending, 'x')],
      y: [coordinates(groups.pending, 'y')],
      z: [coordinates(groups.pending, 'z')],
      'marker.color': [colorsFor(groups.pending, labels)],
      'marker.size': [pointSize(timing.focus_point_size)],
      hovertext: [pendingHoverText],
      hoverinfo: hoverInfo
    }, [trace.pendingPoints]);
  }

  function restyleCentroids(step) {
    const state = states[step.state_position];
    const otherPositions = [];
    for (let index = 0; index < state.centroid_ids.length; index += 1) {
      if (state.centroid_ids[index] !== step.focus_cluster_id) {
        otherPositions.push(index);
      }
    }
    const otherCentroids = otherPositions.map((index) => state.centroids[index]);
    const otherIds = otherPositions.map((index) => state.centroid_ids[index]);
    const focusCentroid = state.centroids[step.focus_centroid_position];
    Plotly.restyle(graph, {
      x: [otherCentroids.map((centroid) => centroid[0])],
      y: [otherCentroids.map((centroid) => centroid[1])],
      z: [otherCentroids.map((centroid) => centroid[2])],
      text: [otherIds.map((clusterId) => `C${clusterId}`)],
      'marker.color': [otherIds.map(clusterColor)]
    }, [trace.otherCentroids]);
    Plotly.restyle(graph, {
      x: [[focusCentroid[0]]],
      y: [[focusCentroid[1]]],
      z: [[focusCentroid[2]]],
      text: [[`C${step.focus_cluster_id}`]],
      'marker.color': [clusterColor(step.focus_cluster_id)]
    }, [trace.focusCentroid]);
  }

  function restyleTransfer(step, labels, targetLabels) {
    const state = states[step.state_position];
    const centroid = state.centroids[step.focus_centroid_position];
    let linePointIndices = [];

    if (step.kind === 'cluster_focus') {
      linePointIndices = splitPointIndices(
        labels,
        targetLabels,
        step.focus_cluster_id
      ).pending;
    }

    if (step.kind === 'cluster_commit') {
      linePointIndices = step.commit_point_indices || [];
    }

    Plotly.restyle(graph, { x: [[]], y: [[]], z: [[]] }, [trace.transferPoint]);

    if (linePointIndices.length === 0) {
      Plotly.restyle(graph, { x: [[]], y: [[]], z: [[]] }, [trace.distanceLine]);
      return;
    }

    const lineX = [];
    const lineY = [];
    const lineZ = [];

    for (const index of linePointIndices) {
      lineX.push(centroid[0], points.x[index], null);
      lineY.push(centroid[1], points.y[index], null);
      lineZ.push(centroid[2], points.z[index], null);
    }

    Plotly.restyle(graph, {
      x: [lineX],
      y: [lineY],
      z: [lineZ],
      'line.color': [clusterColor(step.focus_cluster_id)],
      'line.width': [Number(timing.distance_line_width || 1)]
    }, [trace.distanceLine]);
  }

  function chartElapsedRange(end) {
    const currentElapsed = Number(charts.elapsed[Math.max(0, end - 1)] || 0);
    const earlyWindow = Number(charts.early_elapsed_window_minutes || 50);
    if (currentElapsed <= earlyWindow) {
      return charts.early_elapsed_range || [0, earlyWindow];
    }
    return charts.full_elapsed_range || [0, Math.max(currentElapsed, earlyWindow)];
  }

  function restyleCharts(statePosition) {
    const end = statePosition + 1;
    const elapsed = charts.elapsed.slice(0, end);
    const elapsedRange = chartElapsedRange(end);
    Plotly.restyle(graph, { x: [elapsed], y: [charts.best_fitness.slice(0, end)] }, [trace.best]);
    Plotly.restyle(graph, { x: [elapsed], y: [charts.active_clusters.slice(0, end)] }, [trace.activeClusters]);
    Plotly.restyle(graph, { x: [elapsed], y: [charts.energy_wh.slice(0, end)] }, [trace.energy]);
    Plotly.restyle(graph, { x: [elapsed], y: [charts.cpu_percent.slice(0, end)] }, [trace.cpu]);
    Plotly.restyle(graph, { x: [elapsed], y: [charts.rss_mb.slice(0, end)] }, [trace.ram]);
    Plotly.relayout(graph, {
      'xaxis.range': elapsedRange,
      'xaxis.autorange': false,
      'xaxis2.range': elapsedRange,
      'xaxis2.autorange': false
    });
  }

  function renderStep(stepIndex) {
    currentStepIndex = Math.max(0, Math.min(stepIndex, steps.length - 1));
    const step = steps[currentStepIndex];
    const labels = labelsBefore(currentStepIndex);
    applyCurrentStep(labels, step);
    const targetLabels = targetLabelsForState(currentStepIndex, labels);
    restylePoints(step, labels, targetLabels);
    restyleCentroids(step);
    restyleTransfer(step, labels, targetLabels);
    restyleCharts(step.state_position);
    Plotly.relayout(graph, { 'title.text': titleFor(step) });
    updateControls(step);
  }

  function frameDuration(speedMultiplier) {
    const speed = Math.max(Number(speedMultiplier) || 1, 0.001);
    const rawDurationMs = Number(timing.base_frame_duration_ms || 220) / speed;
    const minimumDurationMs = Math.max(Number(timing.minimum_frame_duration_ms || 0), 0.001);
    return Math.max(rawDurationMs, minimumDurationMs);
  }

  function currentSpeedMultiplier() {
    return Number(timing.speed_multipliers[currentSpeedIndex] || 1);
  }

  function updateSpeedLabel() {
    const label = document.getElementById('ga-animation-speed-label');
    if (label) {
      label.textContent = `${currentSpeedMultiplier()}x`;
    }
    const selector = document.getElementById('ga-animation-speed-select');
    if (selector) {
      selector.value = String(currentSpeedIndex);
    }
  }

  function cancelPlayback() {
    if (playbackFrameRequest !== null) {
      window.cancelAnimationFrame(playbackFrameRequest);
      playbackFrameRequest = null;
    }
    playbackLastTimestamp = null;
    playbackStepRemainderMs = 0;
    isPlaying = false;
  }

  function stopPlayback() {
    cancelPlayback();
    inspectionMode = true;
    renderStep(currentStepIndex);
  }

  function finishPlayback() {
    cancelPlayback();
    inspectionMode = true;
    renderStep(steps.length - 1);
  }

  function playbackTick(timestamp) {
    if (!isPlaying) {
      return;
    }

    if (playbackLastTimestamp === null) {
      playbackLastTimestamp = timestamp;
    }

    const elapsedMs = timestamp - playbackLastTimestamp + playbackStepRemainderMs;
    const durationMs = frameDuration(currentSpeedMultiplier());
    let stepsToAdvance = Math.floor(elapsedMs / durationMs);

    if (stepsToAdvance > 0) {
      stepsToAdvance = Math.min(stepsToAdvance, 100);
      playbackStepRemainderMs = elapsedMs - stepsToAdvance * durationMs;
      playbackLastTimestamp = timestamp;

      if (currentStepIndex >= steps.length - 1) {
        finishPlayback();
        return;
      }

      renderStep(currentStepIndex + stepsToAdvance);
    }

    playbackFrameRequest = window.requestAnimationFrame(playbackTick);
  }

  function play() {
    cancelPlayback();
    if (currentStepIndex >= steps.length - 1) {
      currentStepIndex = 0;
    }
    inspectionMode = false;
    isPlaying = true;
    renderStep(currentStepIndex);
    playbackFrameRequest = window.requestAnimationFrame(playbackTick);
  }

  function restartPlayback() {
    cancelPlayback();
    inspectionMode = true;
    renderStep(0);
  }

  function setSpeedIndex(nextSpeedIndex) {
    const normalizedSpeedIndex = Math.max(
      0,
      Math.min(
        Number(nextSpeedIndex) || 0,
        timing.speed_multipliers.length - 1
      )
    );
    if (normalizedSpeedIndex === currentSpeedIndex) {
      updateSpeedLabel();
      return;
    }
    currentSpeedIndex = normalizedSpeedIndex;
    playbackLastTimestamp = null;
    playbackStepRemainderMs = 0;
    updateSpeedLabel();
  }

  function changeSpeed(delta) {
    setSpeedIndex(currentSpeedIndex + delta);
  }

  function generationSliderIndex(stepIndex) {
    let bestPosition = 0;
    for (let index = 0; index < payload.generation_start_step_indices.length; index += 1) {
      if (payload.generation_start_step_indices[index] <= stepIndex) {
        bestPosition = index;
      }
    }
    return bestPosition;
  }

  function updateControls(step) {
    const slider = document.getElementById('ga-animation-generation-slider');
    const label = document.getElementById('ga-animation-generation-label');
    if (slider) {
      slider.value = String(currentStepIndex);
    }
    if (label) {
      label.textContent = `Generation ${step.generation} | step ${currentStepIndex + 1} of ${steps.length}`;
    }
  }

  function installControls() {
    const controls = document.createElement('div');
    controls.style.display = 'flex';
    controls.style.flexWrap = 'wrap';
    controls.style.alignItems = 'center';
    controls.style.gap = '8px';
    controls.style.margin = '10px 0';
    controls.style.fontFamily = 'Arial, sans-serif';
    const speedOptions = timing.speed_multipliers
      .map((speed, index) => `<option value="${index}">${speed}x</option>`)
      .join('');
    controls.innerHTML = `<button type="button" data-ga-play="1" style="padding:4px 8px;">Play</button><button type="button" data-ga-stop="1" style="padding:4px 8px;">Pause</button><button type="button" data-ga-restart="1" style="padding:4px 8px;">Restart</button><button type="button" data-ga-slower="1" title="Slower" style="padding:4px 8px;">&lt;&lt;</button><button type="button" data-ga-faster="1" title="Faster" style="padding:4px 8px;">&gt;&gt;</button><span id="ga-animation-speed-label" style="min-width:48px;text-align:center;"></span><select id="ga-animation-speed-select" title="Speed" style="padding:4px 6px;">${speedOptions}</select><input id="ga-animation-generation-slider" type="range" min="0" max="${Math.max(steps.length - 1, 0)}" value="0" step="1" style="min-width:260px;flex:1;"><span id="ga-animation-generation-label"></span>`;
    graph.parentNode.insertBefore(controls, graph);
    controls.querySelector('[data-ga-play]').addEventListener('click', play);
    controls.querySelector('[data-ga-stop]').addEventListener('click', stopPlayback);
    controls.querySelector('[data-ga-restart]').addEventListener('click', restartPlayback);
    controls.querySelector('[data-ga-slower]').addEventListener('click', () => changeSpeed(-1));
    controls.querySelector('[data-ga-faster]').addEventListener('click', () => changeSpeed(1));
    controls.querySelector('#ga-animation-speed-select').addEventListener('change', (event) => {
      setSpeedIndex(parseInt(event.target.value, 10));
    });
    controls.querySelector('#ga-animation-generation-slider').addEventListener('input', (event) => {
      const targetStepIndex = Number(event.target.value);
      cancelPlayback();
      inspectionMode = true;
      renderStep(targetStepIndex);
    });
    updateSpeedLabel();
  }

  installControls();
  renderStep(0);
})();
""".replace(
    "__PAYLOAD__",
    json.dumps(
        animation_dashboard_payload,
        ensure_ascii=False,
        separators=(",", ":"),
    ),
)

static_camera = {
    "eye": {
        "x": float(GA_ANIMATION_CAMERA_EYE_X),
        "y": float(GA_ANIMATION_CAMERA_EYE_Y),
        "z": float(GA_ANIMATION_CAMERA_EYE_Z),
    }
}

scene_layout = {
    "xaxis": {"title": "PC1", "range": pca_ranges[0], "autorange": False},
    "yaxis": {"title": "PC2", "range": pca_ranges[1], "autorange": False},
    "zaxis": {"title": "PC3", "range": pca_ranges[2], "autorange": False},
    "aspectmode": "data",
    "camera": static_camera,
}

animation_fig.update_layout(
    title=_layout_title(initial_step),
    height=860,
    uirevision="ga-animation-fixed-view",
    scene=scene_layout,
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": -0.08,
        "xanchor": "left",
        "x": 0,
    },
    updatemenus=[],
    sliders=[],
)

animation_fig.update_xaxes(
    title_text="Elapsed time in minutes",
    range=early_elapsed_range,
    autorange=False,
    row=1,
    col=2,
)
animation_fig.update_yaxes(
    title_text="Fitness",
    range=fitness_range,
    autorange=False,
    row=1,
    col=2,
    secondary_y=False,
)
animation_fig.update_yaxes(
    title_text="Active cluster count",
    range=[0, int(MAX_CLUSTER_COUNT)],
    autorange=False,
    showgrid=False,
    row=1,
    col=2,
    secondary_y=True,
)
animation_fig.update_xaxes(
    title_text="Elapsed time in minutes",
    range=early_elapsed_range,
    autorange=False,
    row=2,
    col=2,
)
animation_fig.update_yaxes(
    title_text="Energy estimate in Wh and CPU usage in percent",
    range=resource_left_range,
    autorange=False,
    row=2,
    col=2,
    secondary_y=False,
)
animation_fig.update_yaxes(
    title_text="RAM RSS in MB",
    range=rss_range,
    autorange=False,
    showgrid=False,
    row=2,
    col=2,
    secondary_y=True,
)

selected_generation_count = len(animation_generation_indices)
generated_frame_count = len(animation_steps)
cluster_focus_frame_count = sum(step.kind == "cluster_focus" for step in animation_steps)
incoming_transfer_frame_count = sum(
    step.kind == "incoming_transfer"
    for step in animation_steps
)
cluster_commit_frame_count = sum(step.kind == "cluster_commit" for step in animation_steps)
estimated_default_playback_seconds = (
    generated_frame_count * default_animation_frame_duration_ms / 1000.0
)

print("Animated GA dashboard prepared.")
print(f"Selected evolutionary generations: {selected_generation_count:,}")
print(f"Generated animation steps: {generated_frame_count:,}")
print(f"Cluster focus steps: {cluster_focus_frame_count:,}")
print(f"Incoming member batch-transfer steps: {incoming_transfer_frame_count:,}")
print(f"Cluster commit batch recolor steps: {cluster_commit_frame_count:,}")
print(f"Default frame duration: {default_animation_frame_duration_ms} ms")
print(f"Estimated default playback duration: {estimated_default_playback_seconds:.2f} s")
print(f"Cluster identity tracking: {GA_ANIMATION_CLUSTER_IDENTITY_TRACKING}")
print(f"Transition mode: {GA_ANIMATION_TRANSITION_MODE}")


Animated GA dashboard prepared.
Selected evolutionary generations: 60
Generated animation steps: 1,286
Cluster focus steps: 643
Incoming member batch-transfer steps: 0
Cluster commit batch recolor steps: 643
Default frame duration: 110.0 ms
Estimated default playback duration: 141.46 s
Cluster identity tracking: membership_overlap_hungarian
Transition mode: incoming_members_only


In [18]:
# =============================================================================
# CELL 12 | Persist optimized GA clustering snapshot artifacts
# =============================================================================

import hashlib
import json
import math
import os
import shutil
import time

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.io as pio


GA_PERSIST_STATIC_HTML = True
GA_PERSIST_ANIMATION_HTML = True
GA_LATEST_USE_HARD_LINKS = True
GA_PLOTLY_VALIDATE_BEFORE_EXPORT = False
GA_ANIMATION_HTML_AUTO_PLAY = False


def _json_safe(
    value: Any,
) -> Any:
    if value is None:
        return None

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    if isinstance(
        value,
        np.integer,
    ):
        return int(
            value
        )

    if isinstance(
        value,
        (
            np.floating,
            float,
        ),
    ):
        value = float(
            value
        )

        return (
            value
            if math.isfinite(
                value
            )
            else None
        )

    if isinstance(
        value,
        (
            np.bool_,
            bool,
        ),
    ):
        return bool(
            value
        )

    if isinstance(
        value,
        np.ndarray,
    ):
        return [
            _json_safe(
                item
            )
            for item
            in value.tolist()
        ]

    if isinstance(
        value,
        pd.Series,
    ):
        return [
            _json_safe(
                item
            )
            for item
            in value.tolist()
        ]

    if isinstance(
        value,
        dict,
    ):
        return {
            str(
                key
            ): _json_safe(
                item
            )
            for (
                key,
                item,
            )
            in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return [
            _json_safe(
                item
            )
            for item
            in value
        ]

    return value


def _sha256_or_none(
    file_path: Path,
) -> str | None:
    if (
        not file_path.exists()
    ):
        return None

    digest = hashlib.sha256()

    with file_path.open(
        "rb"
    ) as file:
        for chunk in iter(
            lambda: file.read(
                1024
                * 1024
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def _file_size_mb(
    file_path: Path,
) -> float:
    if (
        not file_path.exists()
    ):
        return 0.0

    return (
        file_path.stat().st_size
        / (
            1024
            * 1024
        )
    )


def _remove_path(
    path: Path,
) -> None:
    if (
        path.is_symlink()
        or path.is_file()
    ):
        path.unlink(
            missing_ok=True
        )
        return

    if (
        path.exists()
    ):
        shutil.rmtree(
            path
        )


def _write_json_atomic(
    payload: dict[
        str,
        Any,
    ],
    output_path: Path,
) -> None:
    temporary_path = (
        output_path.with_suffix(
            output_path.suffix
            + ".tmp"
        )
    )

    temporary_path.write_text(
        json.dumps(
            _json_safe(
                payload
            ),
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding=(
            "utf-8"
        ),
    )

    os.replace(
        temporary_path,
        output_path,
    )


def _write_dataframe(
    dataframe: pd.DataFrame,
    output_path: Path,
    label: str,
) -> None:
    started_at = (
        time.perf_counter()
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )

    elapsed = (
        time.perf_counter()
        - started_at
    )

    print(
        f"Saved CSV | "
        f"{label} | "
        f"{elapsed:.2f} s | "
        f"{_file_size_mb(output_path):.2f} MB"
    )


def _write_plotly_html(
    figure,
    output_path: Path,
    label: str,
    auto_play: bool,
    post_script: str | None = None,
) -> None:
    started_at = (
        time.perf_counter()
    )

    print(
        f"Writing HTML | "
        f"{label}"
    )

    pio.write_html(
        figure,
        file=(
            output_path
        ),
        include_plotlyjs=(
            "cdn"
        ),
        full_html=True,
        auto_play=(
            auto_play
        ),
        validate=(
            GA_PLOTLY_VALIDATE_BEFORE_EXPORT
        ),
        post_script=(
            post_script
        ),
    )

    elapsed = (
        time.perf_counter()
        - started_at
    )

    print(
        f"Saved HTML | "
        f"{label} | "
        f"{elapsed:.2f} s | "
        f"{_file_size_mb(output_path):.2f} MB"
    )


def _link_or_copy(
    source_path: Path,
    destination_path: Path,
) -> str:
    if (
        GA_LATEST_USE_HARD_LINKS
    ):
        try:
            os.link(
                source_path,
                destination_path,
            )

            return (
                "hard_link"
            )

        except OSError:
            pass

    shutil.copy2(
        source_path,
        destination_path,
    )

    return (
        "copy"
    )


def _replace_latest_directory(
    latest_directory: Path,
    source_files: list[
        Path
    ],
) -> dict[
    str,
    str,
]:
    temporary_directory = (
        latest_directory.parent
        / (
            f"{latest_directory.name}"
            "_temporary"
        )
    )

    _remove_path(
        temporary_directory
    )

    temporary_directory.mkdir(
        parents=True,
        exist_ok=False,
    )

    transfer_modes: dict[
        str,
        str,
    ] = {}

    try:
        for source_path in (
            source_files
        ):
            destination_path = (
                temporary_directory
                / source_path.name
            )

            transfer_modes[
                source_path.name
            ] = _link_or_copy(
                source_path,
                destination_path,
            )

        _remove_path(
            latest_directory
        )

        os.replace(
            temporary_directory,
            latest_directory,
        )

    except Exception:
        _remove_path(
            temporary_directory
        )

        raise

    return (
        transfer_modes
    )


snapshot_created_at_utc = (
    datetime.now(
        timezone.utc
    )
    .isoformat(
        timespec=(
            "microseconds"
        )
    )
    .replace(
        "+00:00",
        "Z",
    )
)

snapshot_directory = (
    GA_CLUSTERING_ROOT_DIRECTORY
    / "snapshots"
    / snapshot_created_at_utc.replace(
        ":",
        "-",
    )
)

snapshot_directory.mkdir(
    parents=True,
    exist_ok=False,
)

latest_directory = (
    GA_CLUSTERING_ROOT_DIRECTORY
    / "latest"
)

print(
    "Snapshot directory: "
    f"{snapshot_directory}"
)


paths = {
    "cluster_assignments": (
        snapshot_directory
        / "ga_cluster_assignments.csv"
    ),
    "cluster_metrics": (
        snapshot_directory
        / "ga_cluster_metrics.csv"
    ),
    "fitness_history": (
        snapshot_directory
        / "ga_fitness_history.csv"
    ),
    "resource_history": (
        snapshot_directory
        / "ga_resource_history.csv"
    ),
    "stopping_progress": (
        snapshot_directory
        / "ga_stopping_progress.csv"
    ),
    "clustering_3d_html": (
        snapshot_directory
        / "ga_clustering_3d.html"
    ),
    "convergence_html": (
        snapshot_directory
        / "ga_convergence_over_time.html"
    ),
    "stopping_progress_html": (
        snapshot_directory
        / "ga_stopping_progress.html"
    ),
    "resource_telemetry_html": (
        snapshot_directory
        / "ga_resource_telemetry.html"
    ),
    "animation_dashboard_html": (
        snapshot_directory
        / "ga_clustering_animation_dashboard.html"
    ),
    "ga_stdout_log": (
        snapshot_directory
        / "ga_stdout.log"
    ),
    "manifest": (
        snapshot_directory
        / "manifest.json"
    ),
}


metadata_columns = [
    "sequence_index",
    "protein_uid",
    "gbseq__accession_version",
    "gbseq__organism",
    "taxonomy__03",
    "taxonomy__04",
    "sequence_length",
]

assignment_columns = [
    column
    for column
    in metadata_columns
    if column
    in plot_df.columns
]

assignment_feature_columns = [
    column
    for column
    in feature_columns
    if column
    in plot_df.columns
]

assignments_output_df = (
    plot_df[
        assignment_columns
        + assignment_feature_columns
    ].copy()
)

assignments_output_df[
    "ga_cluster_label"
] = ga_labels

assignments_output_df[
    "kmeans_cluster_label"
] = kmeans_labels

_write_dataframe(
    assignments_output_df,
    paths[
        "cluster_assignments"
    ],
    "cluster assignments",
)

_write_dataframe(
    metrics_df,
    paths[
        "cluster_metrics"
    ],
    "cluster metrics",
)

_write_dataframe(
    history_df,
    paths[
        "fitness_history"
    ],
    "fitness history",
)


resource_columns = [
    "generation",
    "cumulative_elapsed_seconds",
    "cumulative_elapsed_minutes",
    "generation_elapsed_seconds",
    "process_cpu_seconds_delta",
    "process_cpu_seconds_cumulative",
    "process_cpu_percent_total_capacity",
    "process_cpu_core_percent",
    "process_rss_mb",
    "system_memory_percent",
    "estimated_cpu_energy_joules_delta",
    "estimated_cpu_energy_joules_cumulative",
    "estimated_cpu_energy_wh_cumulative",
]

stopping_columns = [
    "generation",
    "cumulative_elapsed_seconds",
    "cumulative_elapsed_minutes",
    "best_fitness",
    "best_fitness_window_start",
    "fitness_delta_window",
    "fitness_relative_improvement_window",
    "elapsed_seconds_window",
    "process_cpu_seconds_window",
    "estimated_cpu_energy_wh_window",
    "wall_seconds_per_fitness_delta_window",
    "process_cpu_seconds_per_fitness_delta_window",
    "estimated_cpu_energy_joules_per_fitness_delta_window",
]

persisted_resource_columns = [
    column
    for column
    in resource_columns
    if column
    in history_df.columns
]

persisted_stopping_columns = [
    column
    for column
    in stopping_columns
    if column
    in history_df.columns
]

_write_dataframe(
    history_df[
        persisted_resource_columns
    ],
    paths[
        "resource_history"
    ],
    "resource history",
)

_write_dataframe(
    history_df[
        persisted_stopping_columns
    ],
    paths[
        "stopping_progress"
    ],
    "stopping progress",
)


persisted_files: list[
    Path
] = [
    paths[
        "cluster_assignments"
    ],
    paths[
        "cluster_metrics"
    ],
    paths[
        "fitness_history"
    ],
    paths[
        "resource_history"
    ],
    paths[
        "stopping_progress"
    ],
]

if (
    GA_PERSIST_STATIC_HTML
):
    static_html_figures = [
        (
            "clustering 3D",
            fig,
            paths[
                "clustering_3d_html"
            ],
        ),
        (
            "convergence",
            convergence_fig,
            paths[
                "convergence_html"
            ],
        ),
        (
            "stopping progress",
            stopping_fig,
            paths[
                "stopping_progress_html"
            ],
        ),
        (
            "resource telemetry",
            resource_fig,
            paths[
                "resource_telemetry_html"
            ],
        ),
    ]

    for (
        label,
        figure,
        output_path,
    ) in static_html_figures:
        _write_plotly_html(
            figure,
            output_path,
            label,
            auto_play=False,
        )

        persisted_files.append(
            output_path
        )


if (
    GA_PERSIST_ANIMATION_HTML
):
    print(
        "Animation export started."
    )

    print(
        "Animation frame count: "
        f"{len(animation_steps):,}"
    )

    _write_plotly_html(
        animation_fig,
        paths[
            "animation_dashboard_html"
        ],
        "animation dashboard",
        auto_play=(
            GA_ANIMATION_HTML_AUTO_PLAY
        ),
        post_script=(
            globals().get(
                "animation_dashboard_post_script"
            )
        ),
    )

    persisted_files.append(
        paths[
            "animation_dashboard_html"
        ]
    )


ga_verbose_log_path_value = (
    globals().get(
        "ga_verbose_log_path"
    )
)

if (
    ga_verbose_log_path_value
    is not None
):
    ga_verbose_log_source = Path(
        ga_verbose_log_path_value
    )

    if (
        ga_verbose_log_source.exists()
    ):
        shutil.copy2(
            ga_verbose_log_source,
            paths[
                "ga_stdout_log"
            ],
        )

        persisted_files.append(
            paths[
                "ga_stdout_log"
            ]
        )


final_fitness_array = np.asarray(
    final_fitness
).reshape(
    -1
)

best_fitness_value = (
    float(
        final_fitness_array[
            0
        ]
    )
    if len(
        final_fitness_array
    )
    > 0
    else None
)


manifest_payload = {
    "artifact_type": (
        "ga_clustering_snapshot"
    ),
    "snapshot_format_version": (
        "1.4"
    ),
    "snapshot_created_at_utc": (
        snapshot_created_at_utc
    ),
    "input_dataset_name": (
        INPUT_DATASET_NAME
    ),
    "source_pca_kmeans_manifest_sha256": (
        _sha256_or_none(
            PCA_KMEANS_LATEST_DIRECTORY
            / "manifest.json"
        )
    ),
    "source_filtered_datasets_manifest_sha256": (
        _sha256_or_none(
            FILTERED_DATASETS_LATEST_DIRECTORY
            / "manifest.json"
        )
    ),
    "record_count": int(
        X.shape[
            0
        ]
    ),
    "pca_component_count": int(
        PCA_COMPONENT_COUNT
    ),
    "plot_component_count": int(
        PLOT_COMPONENT_COUNT
    ),
    "max_cluster_count": int(
        MAX_CLUSTER_COUNT
    ),
    "min_cluster_count": int(
        MIN_CLUSTER_COUNT
    ),
    "chromosome_bit_count": int(
        chromosome_bit_count
    ),
    "ga_options": (
        GA_OPTIONS
    ),
    "best_fitness": (
        best_fitness_value
    ),
    "elapsed_seconds": float(
        elapsed_seconds
    ),
    "ga_cluster_count": int(
        best_decoded.effective_cluster_count
    ),
    "ga_cluster_sizes": [
        int(
            value
        )
        for value
        in best_decoded.cluster_sizes.tolist()
    ],
    "kmeans_cluster_count": int(
        pd.Series(
            kmeans_labels
        ).nunique()
    ),
    "adjusted_rand_index_ga_vs_kmeans": float(
        comparison_ari
    ),
    "runtime_summary": (
        globals().get(
            "ga_runtime_summary"
        )
    ),
    "stopping_progress_summary": (
        globals().get(
            "ga_stopping_progress_summary"
        )
    ),
    "telemetry_metadata": (
        globals().get(
            "telemetry_metadata"
        )
    ),
    "animation_configuration": {
        "generation_selection_mode": (
            GA_ANIMATION_GENERATION_SELECTION_MODE
        ),
        "improvement_epsilon": float(
            GA_ANIMATION_IMPROVEMENT_EPSILON
        ),
        "selected_generation_count": int(
            len(
                animation_generation_indices
            )
        ),
        "generated_frame_count": int(
            len(
                animation_steps
            )
        ),
        "cluster_focus_frame_count": int(
            cluster_focus_frame_count
        ),
        "incoming_member_recolor_frame_count": int(
            incoming_transfer_frame_count
        ),
        "cluster_commit_frame_count": int(
            cluster_commit_frame_count
        ),
        "max_selected_generations": int(
            GA_ANIMATION_MAX_SELECTED_GENERATIONS
        ),
        "incoming_points_recoloring": (
            "all incoming points are focused together and recolored on cluster_commit"
        ),
        "max_slider_steps": int(
            GA_ANIMATION_MAX_SLIDER_STEPS
        ),
        "dashboard_update_engine": (
            "compact Plotly.restyle post_script without go.Frame payloads"
        ),
        "transition_mode": (
            GA_ANIMATION_TRANSITION_MODE
        ),
        "incoming_point_order": (
            GA_ANIMATION_INCOMING_POINT_ORDER
        ),
        "max_total_frames": (
            GA_ANIMATION_MAX_TOTAL_FRAMES
        ),
        "base_frame_duration_ms": int(
            GA_ANIMATION_BASE_FRAME_DURATION_MS
        ),
        "minimum_frame_duration_ms": float(
            GA_ANIMATION_MIN_FRAME_DURATION_MS
        ),
        "default_speed_multiplier": float(
            GA_ANIMATION_DEFAULT_SPEED_MULTIPLIER
        ),
        "default_frame_duration_ms": int(
            default_animation_frame_duration_ms
        ),
        "frame_transition_ms": int(
            frame_transition_ms
        ),
        "playback_speed_multipliers": [
            float(
                value
            )
            for value
            in globals().get(
                "effective_playback_speed_multipliers",
                GA_ANIMATION_PLAYBACK_SPEED_MULTIPLIERS,
            )
        ],
        "rotate_camera": bool(
            GA_ANIMATION_ROTATE_CAMERA
        ),
        "cluster_identity_tracking": (
            GA_ANIMATION_CLUSTER_IDENTITY_TRACKING
        ),
        "centroid_identity_tracking_detail": (
            "Hungarian assignment maximizing shared members "
            "between consecutive selected generations with "
            "Euclidean centroid distance used as tie breaker"
        ),
        "point_color_mapping": (
            "persistent cluster identifier mapped to one "
            "deterministic HSL color"
        ),
        "opacity_rules": {
            "other_points": float(
                GA_ANIMATION_OTHER_POINT_OPACITY
            ),
            "focused_cluster_points": float(
                GA_ANIMATION_FOCUS_POINT_OPACITY
            ),
            "current_transfer_point": float(
                GA_ANIMATION_TRANSFER_POINT_OPACITY
            ),
            "other_centroids": float(
                GA_ANIMATION_OTHER_CENTROID_OPACITY
            ),
            "focused_centroid": float(
                GA_ANIMATION_FOCUS_CENTROID_OPACITY
            ),
            "distance_line": float(
                GA_ANIMATION_DISTANCE_LINE_OPACITY
            ),
            "distance_line_width": float(
                GA_ANIMATION_DISTANCE_LINE_WIDTH
            ),
        },
        "plotly_validate_before_export": (
            GA_PLOTLY_VALIDATE_BEFORE_EXPORT
        ),
        "animation_html_auto_play": (
            GA_ANIMATION_HTML_AUTO_PLAY
        ),
    },
    "latest_directory_configuration": {
        "use_hard_links": (
            GA_LATEST_USE_HARD_LINKS
        ),
    },
    "files": {
        path.name: {
            "size_mb": (
                _file_size_mb(
                    path
                )
            ),
            "sha256": (
                _sha256_or_none(
                    path
                )
            ),
        }
        for path
        in persisted_files
    },
}


_write_json_atomic(
    manifest_payload,
    paths[
        "manifest"
    ],
)

persisted_files.append(
    paths[
        "manifest"
    ]
)


print(
    "Updating latest directory."
)

latest_transfer_modes = (
    _replace_latest_directory(
        latest_directory=(
            latest_directory
        ),
        source_files=(
            persisted_files
        ),
    )
)

print(
    "Latest directory updated."
)

for (
    file_name,
    transfer_mode,
) in latest_transfer_modes.items():
    print(
        f"Latest | "
        f"{transfer_mode} | "
        f"{file_name}"
    )


print()
print(
    "GA clustering snapshot persisted."
)

print(
    "Snapshot directory: "
    f"{snapshot_directory}"
)

print(
    "Latest directory: "
    f"{latest_directory}"
)

if (
    GA_PERSIST_ANIMATION_HTML
):
    print(
        "Animation dashboard: "
        f"{latest_directory / paths['animation_dashboard_html'].name}"
    )

print(
    "Manifest: "
    f"{latest_directory / paths['manifest'].name}"
)


Snapshot directory: C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\snapshots\2026-06-13T15-52-19.621371Z
Saved CSV | cluster assignments | 0.01 s | 0.19 MB
Saved CSV | cluster metrics | 0.00 s | 0.00 MB
Saved CSV | fitness history | 1.97 s | 78.65 MB
Saved CSV | resource history | 0.99 s | 36.46 MB
Saved CSV | stopping progress | 1.00 s | 41.99 MB
Writing HTML | clustering 3D
Saved HTML | clustering 3D | 0.01 s | 0.13 MB
Writing HTML | convergence
Saved HTML | convergence | 0.07 s | 7.20 MB
Writing HTML | stopping progress
Saved HTML | stopping progress | 0.13 s | 13.57 MB
Writing HTML | resource telemetry
Saved HTML | resource telemetry | 0.22 s | 21.35 MB
Animation export started.
Animation frame count: 1,286
Writing HTML | animation dashboard
Saved HTML | animation dashboard | 0.01 s | 0.75 MB
Updating latest directory.
Latest directory updated.
Latest | hard_link | ga_cluster_assignments.csv
Latest | hard_link | ga_cluster_metrics.csv
Latest | hard_link | ga_fitn

In [19]:
# =============================================================================
# CELL 13 | Lightweight artifact index
# =============================================================================

from pathlib import Path

import pandas as pd

from IPython.display import FileLink, HTML, display


# =============================================================================
# Locate latest persisted snapshot
# =============================================================================

latest_directory = (
    GA_CLUSTERING_ROOT_DIRECTORY
    / "latest"
)

if (
    not latest_directory.exists()
):
    raise FileNotFoundError(
        "Latest GA clustering directory was not found. "
        "Execute CELL 12 before CELL 13. "
        f"Expected directory: {latest_directory}"
    )


# =============================================================================
# Artifact registry
# =============================================================================

artifact_specs = [
    {
        "label": (
            "GA cluster assignments"
        ),
        "file_name": (
            "ga_cluster_assignments.csv"
        ),
        "artifact_type": (
            "CSV"
        ),
    },
    {
        "label": (
            "GA cluster metrics"
        ),
        "file_name": (
            "ga_cluster_metrics.csv"
        ),
        "artifact_type": (
            "CSV"
        ),
    },
    {
        "label": (
            "GA fitness history"
        ),
        "file_name": (
            "ga_fitness_history.csv"
        ),
        "artifact_type": (
            "CSV"
        ),
    },
    {
        "label": (
            "GA resource history"
        ),
        "file_name": (
            "ga_resource_history.csv"
        ),
        "artifact_type": (
            "CSV"
        ),
    },
    {
        "label": (
            "GA stopping progress"
        ),
        "file_name": (
            "ga_stopping_progress.csv"
        ),
        "artifact_type": (
            "CSV"
        ),
    },
    {
        "label": (
            "GA clustering 3D"
        ),
        "file_name": (
            "ga_clustering_3d.html"
        ),
        "artifact_type": (
            "HTML"
        ),
    },
    {
        "label": (
            "GA convergence over time"
        ),
        "file_name": (
            "ga_convergence_over_time.html"
        ),
        "artifact_type": (
            "HTML"
        ),
    },
    {
        "label": (
            "GA stopping progress plot"
        ),
        "file_name": (
            "ga_stopping_progress.html"
        ),
        "artifact_type": (
            "HTML"
        ),
    },
    {
        "label": (
            "GA resource telemetry"
        ),
        "file_name": (
            "ga_resource_telemetry.html"
        ),
        "artifact_type": (
            "HTML"
        ),
    },
    {
        "label": (
            "GA clustering animation dashboard"
        ),
        "file_name": (
            "ga_clustering_animation_dashboard.html"
        ),
        "artifact_type": (
            "HTML animation"
        ),
    },
    {
        "label": (
            "GA snapshot manifest"
        ),
        "file_name": (
            "manifest.json"
        ),
        "artifact_type": (
            "JSON"
        ),
    },
    {
        "label": (
            "GA verbose stdout log"
        ),
        "file_name": (
            "ga_stdout.log"
        ),
        "artifact_type": (
            "LOG"
        ),
    },
]


# =============================================================================
# File metadata helpers
# =============================================================================

def _human_readable_size(
    size_bytes: int,
) -> str:
    size_value = float(
        size_bytes
    )

    size_units = [
        "B",
        "KB",
        "MB",
        "GB",
        "TB",
    ]

    for unit in (
        size_units
    ):
        if (
            size_value
            < 1024.0
            or unit
            == size_units[
                -1
            ]
        ):
            return (
                f"{size_value:.2f} "
                f"{unit}"
            )

        size_value /= (
            1024.0
        )

    return (
        f"{size_value:.2f} TB"
    )


artifact_rows = []

for artifact_spec in (
    artifact_specs
):
    artifact_path = (
        latest_directory
        / artifact_spec[
            "file_name"
        ]
    )

    artifact_exists = (
        artifact_path.exists()
    )

    artifact_rows.append(
        {
            "artifact": (
                artifact_spec[
                    "label"
                ]
            ),
            "type": (
                artifact_spec[
                    "artifact_type"
                ]
            ),
            "file_name": (
                artifact_spec[
                    "file_name"
                ]
            ),
            "exists": (
                artifact_exists
            ),
            "size": (
                _human_readable_size(
                    artifact_path.stat().st_size
                )
                if artifact_exists
                else None
            ),
            "path": (
                str(
                    artifact_path
                )
            ),
        }
    )


artifact_index_df = pd.DataFrame(
    artifact_rows
)

display(
    artifact_index_df
)


# =============================================================================
# Links
# =============================================================================

existing_artifact_count = int(
    artifact_index_df[
        "exists"
    ].sum()
)

print(
    "Persisted artifact directory:"
)

print(
    latest_directory
)

print()

print(
    "Available artifacts:"
)

print(
    f"{existing_artifact_count}"
    f" / "
    f"{len(artifact_specs)}"
)

print()

for artifact_spec in (
    artifact_specs
):
    artifact_path = (
        latest_directory
        / artifact_spec[
            "file_name"
        ]
    )

    if (
        artifact_path.exists()
    ):
        print(
            artifact_spec[
                "label"
            ]
        )

        display(
            FileLink(
                str(
                    artifact_path
                )
            )
        )


# =============================================================================
# Animation guidance
# =============================================================================

animation_dashboard_path = (
    latest_directory
    / "ga_clustering_animation_dashboard.html"
)

if (
    animation_dashboard_path.exists()
):
    animation_dashboard_size = (
        _human_readable_size(
            animation_dashboard_path.stat().st_size
        )
    )

    display(
        HTML(
            (
                "<div style='padding: 12px; "
                "border: 1px solid #cccccc; "
                "border-radius: 8px;'>"
                "<strong>"
                "Animation dashboard ready."
                "</strong>"
                "<br>"
                "Open the HTML link in a separate browser tab. "
                "Inline rendering is intentionally disabled "
                "to keep the notebook responsive."
                "<br>"
                f"Animation HTML size: "
                f"{animation_dashboard_size}"
                "</div>"
            )
        )
    )

,artifact,type,file_name,exists,size,path
0,GA cluster assignments,CSV,ga_cluster_assignments.csv,True,199.00 KB,C:\Programming\Python\pAgo-project\data\04-ana...
1,GA cluster metrics,CSV,ga_cluster_metrics.csv,True,353.00 B,C:\Programming\Python\pAgo-project\data\04-ana...
2,GA fitness history,CSV,ga_fitness_history.csv,True,78.65 MB,C:\Programming\Python\pAgo-project\data\04-ana...
3,GA resource history,CSV,ga_resource_history.csv,True,36.46 MB,C:\Programming\Python\pAgo-project\data\04-ana...
4,GA stopping progress,CSV,ga_stopping_progress.csv,True,41.99 MB,C:\Programming\Python\pAgo-project\data\04-ana...
5,GA clustering 3D,HTML,ga_clustering_3d.html,True,132.30 KB,C:\Programming\Python\pAgo-project\data\04-ana...
6,GA convergence over time,HTML,ga_convergence_over_time.html,True,7.20 MB,C:\Programming\Python\pAgo-project\data\04-ana...
7,GA stopping progress plot,HTML,ga_stopping_progress.html,True,13.57 MB,C:\Programming\Python\pAgo-project\data\04-ana...
8,GA resource telemetry,HTML,ga_resource_telemetry.html,True,21.35 MB,C:\Programming\Python\pAgo-project\data\04-ana...
9,GA clustering animation dashboard,HTML animation,ga_clustering_animation_dashboard.html,True,770.37 KB,C:\Programming\Python\pAgo-project\data\04-ana...


Persisted artifact directory:
C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\latest

Available artifacts:
12 / 12

GA cluster assignments


C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\latest\ga_cluster_assignments.csv

GA cluster metrics


C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\latest\ga_cluster_metrics.csv

...

C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\latest\ga_clustering_animation_dashboard.html

GA snapshot manifest


C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\latest\manifest.json

GA verbose stdout log


C:\Programming\Python\pAgo-project\data\04-analysis\ga_clustering\latest\ga_stdout.log